Load and validate the DEG dataset

In [1]:

# ── Load and validate the DEG dataset ──────────────────────────────────────────
deg_file <- "/mnt/user-uploads/DEG_OSD498_510_radiation_effect.csv"

# Read the CSV; the first column is unnamed (it holds the AGI gene ID)
deg <- read.csv(deg_file, header = TRUE, stringsAsFactors = FALSE)

# Inspect structure
cat("=== Dimensions ===\n")
cat("Rows:", nrow(deg), " Cols:", ncol(deg), "\n")
cat("\n=== Column names ===\n")
print(colnames(deg))
cat("\n=== First 3 rows ===\n")
print(head(deg, 3))
cat("\n=== Last 3 rows ===\n")
print(tail(deg, 3))

# The first column is unnamed — it's the gene ID. Rename it.
colnames(deg)[1] <- "gene_id"
cat("\n=== After renaming first column ===\n")
print(colnames(deg))

# Check the DEG flag column (last column, unnamed)
cat("\n=== DEG flag values ===\n")
print(table(deg[, ncol(deg)]))

# Rename the last column to 'deg_flag'
colnames(deg)[ncol(deg)] <- "deg_flag"
cat("\n=== Final column names ===\n")
print(colnames(deg))

# Validate AGI ID format (AT{1-5}Gnnnnn)
agi_pattern <- "^AT[1-5]G[0-9]{5}$"
valid_ids <- grepl(agi_pattern, deg$gene_id)
cat("\n=== AGI ID validation ===\n")
cat("Valid AGI IDs:", sum(valid_ids), "out of", nrow(deg), "\n")
cat("Invalid examples:", paste(head(deg$gene_id[!valid_ids], 10), collapse=", "), "\n")

# Count DEGs
n_deg <- sum(deg$deg_flag == "yes", na.rm = TRUE)
cat("\n=== DEG count ===\n")
cat("Total DEGs (flag='yes'):", n_deg, "\n")
cat("Non-DEGs (flag='no'):", sum(deg$deg_flag == "no", na.rm = TRUE), "\n")

# Check for missing values in key columns
cat("\n=== Missing values ===\n")
for (col in c("baseMean","log2FoldChange","padj","deg_flag")) {
  cat(col, ":", sum(is.na(deg[[col]])), "NA\n")
}

# Check value ranges
cat("\n=== log2FoldChange range ===\n")
cat("Min:", min(deg$log2FoldChange, na.rm=TRUE), " Max:", max(deg$log2FoldChange, na.rm=TRUE), "\n")
cat("\n=== padj range ===\n")
cat("Min:", min(deg$padj, na.rm=TRUE), " Max:", max(deg$padj, na.rm=TRUE), "\n")

cat("\n=== Dataset loaded and validated successfully ===\n")


=== Dimensions ===
Rows: 23573  Cols: 8 

=== Column names ===
[1] "X"              "baseMean"       "log2FoldChange" "lfcSE"         
[5] "stat"           "pvalue"         "padj"           "X.1"           

=== First 3 rows ===
          X baseMean log2FoldChange     lfcSE     stat   pvalue     padj X.1
1 AT5G60250 373.9543       4.474915 0.3280360 13.64154 2.27e-42 5.34e-38 yes
2 AT3G27630 367.0555       5.769355 0.4284310 13.46624 2.47e-41 2.91e-37 yes
3 AT1G20750 101.5512       6.216640 0.4723034 13.16239 1.44e-39 1.14e-35 yes

=== Last 3 rows ===
              X  baseMean log2FoldChange      lfcSE         stat    pvalue
23571 AT5G53500 517.16162       1.96e-05 0.05362018  0.000365013 0.9997088
23572 AT2G31305  94.10068      -1.11e-05 0.05148909 -0.000215724 0.9998279
23573 AT3G01990  33.49981       1.53e-06 0.11626778  0.000013200 0.9999895
           padj X.1
23571 0.9997936  no
23572 0.9998703  no
23573 0.9999895  no

=== After renaming first column ===
[1] "gene_id"        "bas

Download AtGenExpress atlas from GEO

In [3]:

# ── Download AtGenExpress developmental atlas from GEO ─────────────────────────
# GSE5629 = seedlings and whole plants
# GSE5630 = leaves
# GSE5632 = flowers and pollen
# GSE5633 = shoots and stems
# GSE5634 = roots (separate series)
# GSE5631 = siliques/seeds

suppressPackageStartupMessages({
  library(GEOquery)
})

# Use a local cache directory
geodir <- "/workspace/geo_cache"
dir.create(geodir, showWarnings = FALSE, recursive = TRUE)

# Function to download and extract expression + sample metadata
get_geo_series <- function(gse_id) {
  cat("Downloading", gse_id, "...\n")
  gse <- getGEO(gse_id, destdir = geodir, getGPL = TRUE)
  # getGEO returns a list (one element per platform)
  if (is.list(gse)) gse <- gse[[1]]
  cat("  Class:", class(gse), "\n")
  cat("  Samples:", ncol(gse), "\n")
  cat("  Features:", nrow(gse), "\n")
  return(gse)
}

# Download each series
series_ids <- c("GSE5629", "GSE5630", "GSE5631", "GSE5632", "GSE5633", "GSE5634")
geo_data <- list()

for (sid in series_ids) {
  geo_data[[sid]] <- tryCatch(
    get_geo_series(sid),
    error = function(e) { cat("  ERROR for", sid, ":", conditionMessage(e), "\n"); NULL }
  )
}

cat("\n=== Download summary ===\n")
for (sid in names(geo_data)) {
  if (!is.null(geo_data[[sid]])) {
    cat(sid, ": OK (", ncol(geo_data[[sid]]), "samples,", nrow(geo_data[[sid]]), "features)\n")
  } else {
    cat(sid, ": FAILED\n")
  }
}


Found 1 file(s)

GSE5629_series_matrix.txt.gz

  Class: ExpressionSet 
  Samples: 24 
  Features: 22810 
Found 1 file(s)

GSE5630_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 60 
  Features: 22810 
Found 1 file(s)

GSE5631_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 21 
  Features: 22810 
Found 1 file(s)

GSE5632_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 66 
  Features: 22810 
Found 1 file(s)

GSE5633_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 42 
  Features: 22810 
Found 1 file(s)

GSE5634_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/

Extract expression and metadata from GEO series

In [5]:

# ── Extract expression matrices and sample metadata from all series ────────────
suppressPackageStartupMessages({
  library(Biobase)
})

# Combine all series into one expression matrix + sample metadata
all_expr <- list()
all_meta <- list()

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  if (is.null(gse)) next
  
  # Expression matrix (rows = probes, cols = samples)
  expr <- exprs(gse)
  cat(sid, "expression matrix:", nrow(expr), "x", ncol(expr), "\n")
  
  # Sample metadata from pData
  pdata <- pData(gse)
  cat("  pData columns:", ncol(pdata), "\n")
  
  # The key column is usually 'title' or 'source_name_ch1' or 'characteristics_ch1'
  # Let's look at what's available
  if (sid == "GSE5629") {
    cat("  Sample titles:\n")
    print(pdata$title)
    cat("\n  source_name_ch1:\n")
    if ("source_name_ch1" %in% colnames(pdata)) print(pdata$source_name_ch1)
  }
  
  all_expr[[sid]] <- expr
  all_meta[[sid]] <- pdata
}

cat("\n=== Total samples across all series ===\n")
total_samples <- sum(sapply(all_expr, ncol))
cat(total_samples, "samples\n")


GSE5629 expression matrix: 22810 x 24 
  pData columns: 35 
  Sample titles:
 [1] "ATGE_7_A2"  "ATGE_7_B2"  "ATGE_7_C2"  "ATGE_22_A"  "ATGE_22_B" 
 [6] "ATGE_22_C"  "ATGE_23_A"  "ATGE_23_B"  "ATGE_23_C"  "ATGE_24_A" 
[11] "ATGE_24_B"  "ATGE_24_C"  "ATGE_96_A"  "ATGE_96_B"  "ATGE_96_C" 
[16] "ATGE_97_A"  "ATGE_97_B"  "ATGE_97_C"  "ATGE_100_A" "ATGE_100_B"
[21] "ATGE_100_C" "ATGE_101_A" "ATGE_101_B" "ATGE_101_C"

  source_name_ch1:
 [1] "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0"
[10] "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0"
[19] "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0"
GSE5630 expression matrix: 22810 x 60 
  pData columns: 38 
GSE5631 expression matrix: 22810 x 21 
  pData columns: 36 
GSE5632 expression matrix: 22810 x 66 
  pData columns: 38 
GSE5633 expression matrix: 22810 x 42 
  pData columns: 38 
GSE5634 expression matrix: 22810 x 24 
  pData columns: 35 

=== Total samples across all series ===
237 samples


Examine sample metadata for tissue mapping

In [7]:

# ── Examine sample metadata to build tissue mapping ────────────────────────────
# AtGenExpress uses ATGE sample numbers. We need to map these to tissues.
# Let's look at the characteristics columns for each series

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  pdata <- pData(gse)
  cat("=== ", sid, " ===\n", sep="")
  cat("Titles:\n")
  print(pdata$title)
  
  # Look at characteristics columns
  char_cols <- grep("characteristics", colnames(pdata), value=TRUE)
  for (cc in char_cols) {
    vals <- unique(pdata[[cc]])
    if (length(vals) <= 10) {
      cat("  ", cc, ":\n")
      print(vals)
    }
  }
  
  # Also check description/molecule columns
  desc_cols <- grep("description|source|title|label", colnames(pdata), value=TRUE, ignore.case=TRUE)
  for (dc in desc_cols) {
    vals <- unique(pdata[[dc]])
    if (length(vals) <= 15) {
      cat("  ", dc, ":\n")
      print(vals)
    }
  }
  cat("\n")
}


=== GSE5629 ===
Titles:
 [1] "ATGE_7_A2"  "ATGE_7_B2"  "ATGE_7_C2"  "ATGE_22_A"  "ATGE_22_B" 
 [6] "ATGE_22_C"  "ATGE_23_A"  "ATGE_23_B"  "ATGE_23_C"  "ATGE_24_A" 
[11] "ATGE_24_B"  "ATGE_24_C"  "ATGE_96_A"  "ATGE_96_B"  "ATGE_96_C" 
[16] "ATGE_97_A"  "ATGE_97_B"  "ATGE_97_C"  "ATGE_100_A" "ATGE_100_B"
[21] "ATGE_100_C" "ATGE_101_A" "ATGE_101_B" "ATGE_101_C"
   characteristics_ch1 :
[1] "Tissue: seedling, green parts"                                                
[2] "Tissue: developmental drift; whole plant after transition, but before bolting"
   source_name_ch1 :
[1] "Col-0"
   label_ch1 :
[1] "biotin"
   description :
[1] "Photoperiod: continuous light"
   description.1 :
[1] "Relative humidity: 65%"         "Substrate: 0.5x GM"            
[3] "Substrate: 0.5x GM, 5% sucrose"
   description.2 :
[1] "Substrate: soil"   "Temperature: 20oC"
   description.3 :
[1] "Temperature: 23oC"                                                                                          
[2] "Tissu

Extract tissue labels from all samples

In [9]:

# ── Build tissue mapping for all samples ───────────────────────────────────────
# Extract tissue info from characteristics_ch1 columns for each series

extract_tissue <- function(pdata) {
  # Find all characteristics columns
  char_cols <- grep("characteristics_ch1", colnames(pdata), value=TRUE)
  tissue <- rep(NA_character_, nrow(pdata))
  
  for (cc in char_cols) {
    vals <- pdata[[cc]]
    # Look for "Tissue: ..." pattern
    tissue_matches <- grepl("Tissue:", vals)
    if (any(tissue_matches)) {
      extracted <- sub(".*Tissue:\\s*", "", vals)
      extracted <- sub("\\s*;.*", "", extracted)  # remove anything after ;
      extracted[!tissue_matches] <- NA
      # Fill in where we don't have tissue yet
      need <- is.na(tissue)
      tissue[need] <- extracted[need]
    }
  }
  return(tissue)
}

# Build combined sample metadata
all_samples <- data.frame(
  sample_id = character(),
  series = character(),
  tissue_raw = character(),
  stringsAsFactors = FALSE
)

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  pdata <- pData(gse)
  tissue <- extract_tissue(pdata)
  
  df <- data.frame(
    sample_id = pdata$title,
    series = sid,
    tissue_raw = tissue,
    stringsAsFactors = FALSE
  )
  all_samples <- rbind(all_samples, df)
}

cat("=== Total samples ===\n")
cat(nrow(all_samples), "\n")
cat("\n=== Samples with tissue annotation ===\n")
cat(sum(!is.na(all_samples$tissue_raw)), "out of", nrow(all_samples), "\n")

cat("\n=== Unique tissue labels ===\n")
tissue_table <- sort(table(all_samples$tissue_raw), decreasing=TRUE)
print(tissue_table)

cat("\n=== Samples missing tissue ===\n")
missing <- all_samples[is.na(all_samples$tissue_raw), ]
print(missing[, c("sample_id", "series")])


=== Total samples ===
237 

=== Samples with tissue annotation ===
237 out of 237 

=== Unique tissue labels ===

shoot apex, inflorescence (after bolting) 
                                       24 
               flower stage 12 equivalent 
                                       21 
                                     root 
                                       15 
                    seedling, green parts 
                                       15 
                      developmental drift 
                                        9 
           veg rosette (for phase change) 
                                        9 
                                    roots 
                                        6 
                        rosette leaf # 12 
                                        6 
                rosette leaf #4, 1cm long 
                                        6 
                         rosette leaf # 4 
                                        4 
                          

Build hierarchical tissue mapping

In [11]:

# ── Build hierarchical tissue mapping: broad_organ → sub_tissue ────────────────

# Map each raw tissue label to (broad_organ, sub_tissue)
build_tissue_map <- function(raw_tissue) {
  t <- tolower(trimws(raw_tissue))
  
  # ROOT
  if (grepl("root", t)) {
    return(list(broad = "Root", sub = "root"))
  }
  
  # SEEDLING
  if (grepl("seedling", t)) {
    return(list(broad = "Seedling", sub = "seedling_green_parts"))
  }
  if (grepl("developmental drift", t)) {
    return(list(broad = "Seedling", sub = "whole_plant_pre_bolting"))
  }
  
  # LEAF / SHOOT
  if (grepl("cotyledon", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cotyledon"))
  }
  if (grepl("hypocotyl", t)) {
    return(list(broad = "Leaf_Shoot", sub = "hypocotyl"))
  }
  if (grepl("senescing", t)) {
    return(list(broad = "Leaf_Shoot", sub = "senescing_leaf"))
  }
  if (grepl("rosette leaf", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("^leaf", t) || grepl("leaves 1", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("cauline", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cauline_leaf"))
  }
  if (grepl("veg rosette", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_vegetative"))
  }
  if (grepl("shoot apex, vegetative", t) && !grepl("inflorescence", t)) {
    if (grepl("young leaves", t)) return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative_with_leaves"))
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative"))
  }
  if (grepl("shoot apex, transition", t)) {
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_transition"))
  }
  if (grepl("stem", t) || grepl("1st node", t)) {
    return(list(broad = "Leaf_Shoot", sub = "stem"))
  }
  
  # FLOWER
  if (grepl("pollen", t)) {
    return(list(broad = "Flower", sub = "mature_pollen"))
  }
  if (grepl("carpel", t)) {
    return(list(broad = "Flower", sub = "carpel"))
  }
  if (grepl("petal", t)) {
    return(list(broad = "Flower", sub = "petal"))
  }
  if (grepl("sepal", t)) {
    return(list(broad = "Flower", sub = "sepal"))
  }
  if (grepl("stamen", t)) {
    return(list(broad = "Flower", sub = "stamen"))
  }
  if (grepl("pedicel", t)) {
    return(list(broad = "Flower", sub = "pedicel"))
  }
  if (grepl("flower stage 12 equivalent", t) || grepl("^flower$", t)) {
    return(list(broad = "Flower", sub = "flower_whole"))
  }
  if (grepl("flowers stage 9", t)) {
    return(list(broad = "Flower", sub = "flower_stage_9"))
  }
  if (grepl("flowers stage 10", t)) {
    return(list(broad = "Flower", sub = "flower_stage_10_11"))
  }
  if (grepl("flowers stage 12", t)) {
    return(list(broad = "Flower", sub = "flower_stage_12"))
  }
  if (grepl("flowers stage 15", t)) {
    return(list(broad = "Flower", sub = "flower_stage_15"))
  }
  if (grepl("shoot apex, inflorescence", t)) {
    return(list(broad = "Flower", sub = "inflorescence_apex"))
  }
  
  # SEED / SILIQUE
  if (grepl("silique", t)) {
    if (grepl("stage 3", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_3"))
    if (grepl("stage 4", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_4"))
    if (grepl("stage 5", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_5"))
    return(list(broad = "Seed_Silique", sub = "silique"))
  }
  if (grepl("seeds?", t)) {
    if (grepl("stage 6", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_6"))
    if (grepl("stage 7", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_7"))
    if (grepl("stage 8", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_8"))
    if (grepl("stage 9", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_9"))
    if (grepl("stage 10", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_10"))
    return(list(broad = "Seed_Silique", sub = "seed"))
  }
  
  # Fallback
  return(list(broad = "Other", sub = "unknown"))
}

# Apply mapping
broad_organs <- character(nrow(all_samples))
sub_tissues <- character(nrow(all_samples))

for (i in seq_len(nrow(all_samples))) {
  mapping <- build_tissue_map(all_samples$tissue_raw[i])
  broad_organs[i] <- mapping$broad
  sub_tissues[i] <- mapping$sub
}

all_samples$broad_organ <- broad_organs
all_samples$sub_tissue <- sub_tissues

cat("=== Broad organ distribution ===\n")
print(sort(table(all_samples$broad_organ), decreasing=TRUE))

cat("\n=== Sub-tissue distribution ===\n")
print(sort(table(all_samples$sub_tissue), decreasing=TRUE))

cat("\n=== Hierarchical view ===\n")
for (bo in unique(all_samples$broad_organ)) {
  subs <- unique(all_samples$sub_tissue[all_samples$broad_organ == bo])
  cat(bo, ":\n")
  for (s in subs) {
    n <- sum(all_samples$broad_organ == bo & all_samples$sub_tissue == s)
    cat("  ", s, " (", n, " samples)\n", sep="")
  }
}


=== Broad organ distribution ===

      Flower   Leaf_Shoot Seed_Silique     Seedling         Root 
          90           78           24           24           21 

=== Sub-tissue distribution ===

                     rosette_leaf                      flower_whole 
                               42                                24 
               inflorescence_apex                              root 
                               24                                21 
             seedling_green_parts                           silique 
                               15                                15 
               rosette_vegetative           whole_plant_pre_bolting 
                                9                                 9 
                           carpel                             petal 
                                6                                 6 
                            sepal                            stamen 
                                6        

Fix tissue mapping for seed stages

In [13]:

# ── Fix: check "seeds" BEFORE "silique" in the mapping ─────────────────────────
build_tissue_map_fixed <- function(raw_tissue) {
  t <- tolower(trimws(raw_tissue))
  
  # ROOT
  if (grepl("root", t)) {
    return(list(broad = "Root", sub = "root"))
  }
  
  # SEEDLING
  if (grepl("seedling", t)) {
    return(list(broad = "Seedling", sub = "seedling_green_parts"))
  }
  if (grepl("developmental drift", t)) {
    return(list(broad = "Seedling", sub = "whole_plant_pre_bolting"))
  }
  
  # LEAF / SHOOT
  if (grepl("cotyledon", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cotyledon"))
  }
  if (grepl("hypocotyl", t)) {
    return(list(broad = "Leaf_Shoot", sub = "hypocotyl"))
  }
  if (grepl("senescing", t)) {
    return(list(broad = "Leaf_Shoot", sub = "senescing_leaf"))
  }
  if (grepl("rosette leaf", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("^leaf", t) || grepl("leaves 1", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("cauline", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cauline_leaf"))
  }
  if (grepl("veg rosette", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_vegetative"))
  }
  if (grepl("shoot apex, vegetative", t) && !grepl("inflorescence", t)) {
    if (grepl("young leaves", t)) return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative_with_leaves"))
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative"))
  }
  if (grepl("shoot apex, transition", t)) {
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_transition"))
  }
  if (grepl("stem", t) || grepl("1st node", t)) {
    return(list(broad = "Leaf_Shoot", sub = "stem"))
  }
  
  # FLOWER
  if (grepl("pollen", t)) {
    return(list(broad = "Flower", sub = "mature_pollen"))
  }
  if (grepl("carpel", t)) {
    return(list(broad = "Flower", sub = "carpel"))
  }
  if (grepl("petal", t)) {
    return(list(broad = "Flower", sub = "petal"))
  }
  if (grepl("sepal", t)) {
    return(list(broad = "Flower", sub = "sepal"))
  }
  if (grepl("stamen", t)) {
    return(list(broad = "Flower", sub = "stamen"))
  }
  if (grepl("pedicel", t)) {
    return(list(broad = "Flower", sub = "pedicel"))
  }
  if (grepl("flower stage 12 equivalent", t) || grepl("^flower$", t)) {
    return(list(broad = "Flower", sub = "flower_whole"))
  }
  if (grepl("flowers stage 9", t)) {
    return(list(broad = "Flower", sub = "flower_stage_9"))
  }
  if (grepl("flowers stage 10", t)) {
    return(list(broad = "Flower", sub = "flower_stage_10_11"))
  }
  if (grepl("flowers stage 12", t)) {
    return(list(broad = "Flower", sub = "flower_stage_12"))
  }
  if (grepl("flowers stage 15", t)) {
    return(list(broad = "Flower", sub = "flower_stage_15"))
  }
  if (grepl("shoot apex, inflorescence", t)) {
    return(list(broad = "Flower", sub = "inflorescence_apex"))
  }
  
  # SEED / SILIQUE — check "seeds, stage" BEFORE "silique" 
  # because "seeds, stage X, w/o siliques" contains the word "siliques"
  if (grepl("^seeds?,", t)) {
    if (grepl("stage 6", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_6"))
    if (grepl("stage 7", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_7"))
    if (grepl("stage 8", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_8"))
    if (grepl("stage 9", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_9"))
    if (grepl("stage 10", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_10"))
    return(list(broad = "Seed_Silique", sub = "seed"))
  }
  if (grepl("silique", t)) {
    if (grepl("stage 3", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_3"))
    if (grepl("stage 4", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_4"))
    if (grepl("stage 5", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_5"))
    return(list(broad = "Seed_Silique", sub = "silique"))
  }
  
  # Fallback
  return(list(broad = "Other", sub = "unknown"))
}

# Re-apply mapping
for (i in seq_len(nrow(all_samples))) {
  mapping <- build_tissue_map_fixed(all_samples$tissue_raw[i])
  all_samples$broad_organ[i] <- mapping$broad
  all_samples$sub_tissue[i] <- mapping$sub
}

cat("=== Fixed broad organ distribution ===\n")
print(sort(table(all_samples$broad_organ), decreasing=TRUE))

cat("\n=== Fixed sub-tissue distribution ===\n")
print(sort(table(all_samples$sub_tissue), decreasing=TRUE))

cat("\n=== Fixed hierarchical view ===\n")
for (bo in sort(unique(all_samples$broad_organ))) {
  subs <- sort(unique(all_samples$sub_tissue[all_samples$broad_organ == bo]))
  cat(bo, ":\n")
  for (s in subs) {
    n <- sum(all_samples$broad_organ == bo & all_samples$sub_tissue == s)
    cat("  ", s, " (", n, " samples)\n", sep="")
  }
}

# Check for any "Other/unknown"
cat("\n=== Unmapped samples ===\n")
other <- all_samples[all_samples$broad_organ == "Other", ]
print(other[, c("sample_id", "series", "tissue_raw")])


=== Fixed broad organ distribution ===

      Flower   Leaf_Shoot Seed_Silique     Seedling         Root 
          90           78           24           24           21 

=== Fixed sub-tissue distribution ===

                     rosette_leaf                      flower_whole 
                               42                                24 
               inflorescence_apex                              root 
                               24                                21 
             seedling_green_parts                rosette_vegetative 
                               15                                 9 
          whole_plant_pre_bolting                            carpel 
                                9                                 6 
                            petal                             sepal 
                                6                                 6 
                           stamen                              stem 
                             

Combine expression and map probes to AGI IDs

In [15]:

# ── Combine all expression matrices and map probes to AGI IDs ──────────────────
suppressPackageStartupMessages({
  library(ath1121501.db)
  library(AnnotationDbi)
})

# Combine expression matrices from all series
# Each series has the same 22810 probes (ATH1 platform)
combined_expr <- NULL
sample_order <- c()  # track which sample belongs to which series

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  expr <- exprs(gse)
  if (is.null(combined_expr)) {
    combined_expr <- expr
  } else {
    # cbind — probes should be in the same order (same platform)
    combined_expr <- cbind(combined_expr, expr)
  }
  sample_order <- c(sample_order, rep(sid, ncol(expr)))
}

cat("=== Combined expression matrix ===\n")
cat("Dimensions:", nrow(combined_expr), "probes x", ncol(combined_expr), "samples\n")

# Verify sample order matches all_samples
# all_samples was built in the same order as geo_data, so they should match
cat("Sample order matches:", all(sample_order == all_samples$series), "\n")

# Map column names of combined_expr to all_samples
colnames(combined_expr) <- all_samples$sample_id

# ── Map ATH1 probe IDs to AGI gene IDs ─────────────────────────────────────────
# Get the probe-to-gene mapping from ath1121501.db
probe_ids <- rownames(combined_expr)
cat("\n=== Probe IDs ===\n")
cat("Total probes:", length(probe_ids), "\n")
cat("Examples:", paste(head(probe_ids, 5), collapse=", "), "\n")

# Get AGI mappings
agi_map <- AnnotationDbi::select(ath1121501.db, 
                                  keys = probe_ids, 
                                  columns = c("PROBEID", "ENTREZID", "SYMBOL", "GENENAME"),
                                  keytype = "PROBEID")

# Also get the Arabidopsis locus IDs (AGI)
# ath1121501.db maps to ENTREZID; we need AGI. Let's check available columns
cat("\n=== Available columns in ath1121501.db ===\n")
print(columns(ath1121501.db))

# Try to get AGI directly
agi_map2 <- AnnotationDbi::select(ath1121501.db,
                                   keys = probe_ids,
                                   columns = c("PROBEID", "ENTREZID", "SYMBOL"),
                                   keytype = "PROBEID")

cat("\n=== Probe mapping summary ===\n")
cat("Probes with ENTREZID:", sum(!is.na(agi_map2$ENTREZID)), "\n")
cat("Probes with SYMBOL:", sum(!is.na(agi_map2$SYMBOL)), "\n")

# The SYMBOL in ath1121501.db often contains the AGI locus ID
cat("\n=== SYMBOL examples ===\n")
print(head(agi_map2$SYMBOL[!is.na(agi_map2$SYMBOL)], 20))


=== Combined expression matrix ===
Dimensions: 22810 probes x 237 samples
Sample order matches: TRUE 

=== Probe IDs ===
Total probes: 22810 
Examples: 244901_at, 244902_at, 244903_at, 244904_at, 244905_at 
'select()' returned 1:many mapping between keys and columns


=== Available columns in ath1121501.db ===
 [1] "ARACYC"       "ARACYCENZYME" "ENTREZID"     "ENZYME"       "EVIDENCE"    
 [6] "EVIDENCEALL"  "GENENAME"     "GO"           "GOALL"        "ONTOLOGY"    
[11] "ONTOLOGYALL"  "PATH"         "PMID"         "PROBEID"      "REFSEQ"      
[16] "SYMBOL"       "TAIR"        
'select()' returned 1:many mapping between keys and columns


=== Probe mapping summary ===
Probes with ENTREZID: 32641 
Probes with SYMBOL: 24493 

=== SYMBOL examples ===
 [1] "ORF25"   "NAD4L"   "ORF149"  "ORF275"  "ORF122C" "ORF240A" "ORF120" 
 [8] "ORF107D" "ORF100A" "ORF119"  "ORF170"  "CCB382"  "ORF121B" "ORF107E"
[15] "ORF158"  "ORF187"  "ORF184"  "ORF106D" "CCB203"  "NAD3"   


Get AGI locus IDs from TAIR column

In [17]:

# ── Get AGI locus IDs from the TAIR column ─────────────────────────────────────
agi_map_tair <- AnnotationDbi::select(ath1121501.db,
                                       keys = probe_ids,
                                       columns = c("PROBEID", "TAIR", "SYMBOL", "GENENAME"),
                                       keytype = "PROBEID")

cat("=== TAIR column examples ===\n")
print(head(agi_map_tair$TAIR[!is.na(agi_map_tair$TAIR)], 20))

cat("\n=== TAIR mapping summary ===\n")
cat("Total probes:", nrow(agi_map_tair), "\n")
cat("Probes with TAIR:", sum(!is.na(agi_map_tair$TAIR)), "\n")
cat("Unique TAIR IDs:", length(unique(agi_map_tair$TAIR[!is.na(agi_map_tair$TAIR)])), "\n")

# Check if TAIR values look like AGI IDs (AT{1-5}Gnnnnn)
tair_vals <- agi_map_tair$TAIR[!is.na(agi_map_tair$TAIR)]
agi_like <- grepl("^AT[1-5]G[0-9]{5}", tair_vals)
cat("TAIR values matching AGI pattern:", sum(agi_like), "out of", length(tair_vals), "\n")

# Show some non-AGI TAIR values
non_agi <- tair_vals[!agi_like]
cat("Non-AGI TAIR examples:", paste(head(non_agi, 20), collapse=", "), "\n")


'select()' returned 1:many mapping between keys and columns

=== TAIR column examples ===
 [1] "ATMG00640" "ATMG00650" "ATMG00660" "ATMG00670" "ATMG00680" "ATMG00690"
 [7] "ATMG00710" "ATMG00720" "ATMG00740" "ATMG00750" "ATMG00820" "ATMG00830"
[13] "ATMG00840" "ATMG00850" "ATMG00860" "ATMG00880" "ATMG00870" "ATMG00890"
[19] "AT2G07768" "ATMG00960"

=== TAIR mapping summary ===
Total probes: 34760 
Probes with TAIR: 33143 
Unique TAIR IDs: 21891 
TAIR values matching AGI pattern: 32890 out of 33143 
Non-AGI TAIR examples: ATMG00640, ATMG00650, ATMG00660, ATMG00670, ATMG00680, ATMG00690, ATMG00710, ATMG00720, ATMG00740, ATMG00750, ATMG00820, ATMG00830, ATMG00840, ATMG00850, ATMG00860, ATMG00880, ATMG00870, ATMG00890, ATMG00960, ATMG00990 


Collapse probes to genes and build tissue expression matrix

In [19]:

# ── Collapse probes to genes and build tissue-level expression matrix ──────────

# Keep only nuclear AGI IDs (AT{1-5}Gnnnnn), drop organellar (ATCG/ATMG)
agi_map_clean <- agi_map_tair[!is.na(agi_map_tair$TAIR), ]
agi_map_clean <- agi_map_clean[grepl("^AT[1-5]G[0-9]{5}", agi_map_clean$TAIR), ]

# Some probes map to multiple genes; keep first mapping per probe
agi_map_clean <- agi_map_clean[!duplicated(agi_map_clean$PROBEID), ]

cat("=== Clean probe-to-gene mapping ===\n")
cat("Probes with nuclear AGI:", nrow(agi_map_clean), "\n")
cat("Unique genes:", length(unique(agi_map_clean$TAIR)), "\n")

# Subset expression matrix to mapped probes
mapped_probes <- agi_map_clean$PROBEID
expr_mapped <- combined_expr[mapped_probes, , drop = FALSE]
rownames(expr_mapped) <- agi_map_clean$TAIR[match(mapped_probes, agi_map_clean$PROBEID)]

# If multiple probes map to the same gene, take the one with highest mean expression
gene_ids <- rownames(expr_mapped)
dup_genes <- unique(gene_ids[duplicated(gene_ids)])
cat("\n=== Genes with multiple probes ===\n")
cat("Duplicated genes:", length(dup_genes), "\n")

if (length(dup_genes) > 0) {
  # For each duplicated gene, keep the probe with highest mean expression
  keep_rows <- seq_len(nrow(expr_mapped))
  mean_expr <- rowMeans(expr_mapped, na.rm = TRUE)
  
  for (g in dup_genes) {
    idx <- which(gene_ids == g)
    best <- idx[which.max(mean_expr[idx])]
    keep_rows <- setdiff(keep_rows, setdiff(idx, best))
  }
  
  expr_mapped <- expr_mapped[keep_rows, , drop = FALSE]
  cat("After deduplication:", nrow(expr_mapped), "genes\n")
}

cat("\n=== Final gene-level expression matrix ===\n")
cat("Dimensions:", nrow(expr_mapped), "genes x", ncol(expr_mapped), "samples\n")
cat("Gene ID examples:", paste(head(rownames(expr_mapped), 5), collapse=", "), "\n")

# ── Build tissue-level expression matrix (mean per sub_tissue) ─────────────────
# For each sub_tissue, average expression across all samples of that tissue
sub_tissues <- unique(all_samples$sub_tissue)
sub_tissue_expr <- matrix(NA, nrow = nrow(expr_mapped), ncol = length(sub_tissues))
rownames(sub_tissue_expr) <- rownames(expr_mapped)
colnames(sub_tissue_expr) <- sub_tissues

for (st in sub_tissues) {
  samples_st <- all_samples$sample_id[all_samples$sub_tissue == st]
  # Make sure these columns exist
  samples_st <- intersect(samples_st, colnames(expr_mapped))
  if (length(samples_st) > 0) {
    sub_tissue_expr[, st] <- rowMeans(expr_mapped[, samples_st, drop = FALSE], na.rm = TRUE)
  }
}

cat("\n=== Sub-tissue expression matrix ===\n")
cat("Dimensions:", nrow(sub_tissue_expr), "genes x", ncol(sub_tissue_expr), "sub-tissues\n")

# Also build broad-organ level (mean per broad_organ)
broad_organs <- unique(all_samples$broad_organ)
broad_organ_expr <- matrix(NA, nrow = nrow(expr_mapped), ncol = length(broad_organs))
rownames(broad_organ_expr) <- rownames(expr_mapped)
colnames(broad_organ_expr) <- broad_organs

for (bo in broad_organs) {
  samples_bo <- all_samples$sample_id[all_samples$broad_organ == bo]
  samples_bo <- intersect(samples_bo, colnames(expr_mapped))
  if (length(samples_bo) > 0) {
    broad_organ_expr[, bo] <- rowMeans(expr_mapped[, samples_bo, drop = FALSE], na.rm = TRUE)
  }
}

cat("\n=== Broad-organ expression matrix ===\n")
cat("Dimensions:", nrow(broad_organ_expr), "genes x", ncol(broad_organ_expr), "broad organs\n")
print(head(broad_organ_expr[, 1:5], 3))

# Save the atlas data for reuse
saveRDS(list(
  sub_tissue_expr = sub_tissue_expr,
  broad_organ_expr = broad_organ_expr,
  all_samples = all_samples,
  agi_map_clean = agi_map_clean
), "/workspace/atgenexpress_atlas.rds")

cat("\n=== Atlas saved to /workspace/atgenexpress_atlas.rds ===\n")


=== Clean probe-to-gene mapping ===
Probes with nuclear AGI: 21045 
Unique genes: 20833 

=== Genes with multiple probes ===
Duplicated genes: 196 
After deduplication: 20833 genes

=== Final gene-level expression matrix ===
Dimensions: 20833 genes x 237 samples
Gene ID examples: AT2G07768, AT2G07751, AT2G07674, AT2G07776, AT2G07732 

=== Sub-tissue expression matrix ===
Dimensions: 20833 genes x 33 sub-tissues

=== Broad-organ expression matrix ===
Dimensions: 20833 genes x 5 broad organs
           Seedling Leaf_Shoot      Root    Flower Seed_Silique
AT2G07768  10.02431   9.630731  14.64738  14.94571     20.86445
AT2G07751 124.18204 124.371355 149.83692 173.57268    157.99836
AT2G07674  18.04145  20.036461  13.60768  26.28298     13.35861

=== Atlas saved to /workspace/atgenexpress_atlas.rds ===


Compute Tau index and assign DEGs to tissues

In [21]:

# ── Compute Tau tissue-specificity index and assign DEGs to tissues ────────────
# Tau = Σ(1 − xᵢ/max(x)) / (n − 1)
# where xᵢ = expression in tissue i, n = number of tissues
# Range: 0 (ubiquitous) to 1 (perfectly tissue-specific)

# Load atlas data
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
sub_tissue_expr <- atlas$sub_tissue_expr
broad_organ_expr <- atlas$broad_organ_expr

# ── Tau at sub-tissue level ────────────────────────────────────────────────────
compute_tau <- function(expr_matrix) {
  # expr_matrix: genes (rows) x tissues (cols)
  # Returns vector of Tau scores per gene
  
  # Handle any negative or zero values — shift to positive
  expr_matrix[expr_matrix < 0] <- 0
  
  # For each gene, compute Tau
  tau <- apply(expr_matrix, 1, function(x) {
    mx <- max(x, na.rm = TRUE)
    if (mx == 0 || is.na(mx)) return(NA)
    n <- sum(!is.na(x))
    if (n <= 1) return(NA)
    sum(1 - (x / mx), na.rm = TRUE) / (n - 1)
  })
  
  return(tau)
}

# Compute Tau at both levels
tau_subtissue <- compute_tau(sub_tissue_expr)
tau_broad <- compute_tau(broad_organ_expr)

cat("=== Tau index summary (sub-tissue level) ===\n")
cat("Genes with valid Tau:", sum(!is.na(tau_subtissue)), "\n")
cat("Mean:", round(mean(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Median:", round(median(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Min:", round(min(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Max:", round(max(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Quantiles:\n")
print(round(quantile(tau_subtissue, na.rm=TRUE), 4))

cat("\n=== Tau index summary (broad-organ level) ===\n")
cat("Genes with valid Tau:", sum(!is.na(tau_broad)), "\n")
cat("Mean:", round(mean(tau_broad, na.rm=TRUE), 4), "\n")
cat("Median:", round(median(tau_broad, na.rm=TRUE), 4), "\n")
cat("Quantiles:\n")
print(round(quantile(tau_broad, na.rm=TRUE), 4))

# ── Assign each gene to its predominant tissue ─────────────────────────────────
# For tissue-specific genes (Tau >= threshold), assign to tissue with highest expression
# For constitutive genes (Tau < threshold), label as "constitutive"

TAU_THRESHOLD <- 0.6  # standard threshold from literature

# Sub-tissue assignment
assign_tissue <- function(expr_matrix, tau_values, threshold) {
  # For each gene, find the tissue with max expression
  predominant_tissue <- colnames(expr_matrix)[apply(expr_matrix, 1, function(x) {
    if (all(is.na(x)) || max(x, na.rm=TRUE) == 0) return(NA)
    which.max(x)
  })]
  
  # Classification: tissue-specific vs constitutive
  specificity <- ifelse(tau_values >= threshold, "tissue_specific", "constitutive")
  specificity[is.na(tau_values)] <- "unannotated"
  
  return(list(predominant_tissue = predominant_tissue, specificity = specificity))
}

# Sub-tissue level
sub_assignment <- assign_tissue(sub_tissue_expr, tau_subtissue, TAU_THRESHOLD)

# Broad-organ level  
broad_assignment <- assign_tissue(broad_organ_expr, tau_broad, TAU_THRESHOLD)

# Build the gene annotation table
gene_annotation <- data.frame(
  gene_id = rownames(sub_tissue_expr),
  tau_subtissue = tau_subtissue[rownames(sub_tissue_expr)],
  tau_broad = tau_broad[rownames(sub_tissue_expr)],
  predominant_subtissue = sub_assignment$predominant_tissue,
  predominant_broad_organ = broad_assignment$predominant_tissue,
  specificity = sub_assignment$specificity,
  stringsAsFactors = FALSE
)

# For broad organ, map sub-tissue to broad organ
sub_to_broad <- unique(all_samples[, c("sub_tissue", "broad_organ")])
names(sub_to_broad) <- c("sub_tissue", "broad_organ")
gene_annotation$predominant_broad_organ <- sub_to_broad$broad_organ[
  match(gene_annotation$predominant_subtissue, sub_to_broad$sub_tissue)]

cat("\n=== Specificity classification ===\n")
print(table(gene_annotation$specificity))

cat("\n=== Predominant broad organ distribution ===\n")
print(table(gene_annotation$predominant_broad_organ, useNA = "ifany"))

cat("\n=== Predominant sub-tissue distribution (top 15) ===\n")
print(sort(table(gene_annotation$predominant_subtissue, useNA = "ifany"), decreasing=TRUE)[1:15])

# ── Merge with DEG data ────────────────────────────────────────────────────────
# Load the DEG data (re-read to get clean version)
deg <- read.csv("/mnt/user-uploads/DEG_OSD498_510_radiation_effect.csv", 
                header = TRUE, stringsAsFactors = FALSE)
colnames(deg)[1] <- "gene_id"
colnames(deg)[ncol(deg)] <- "deg_flag"

# Merge
deg_annotated <- merge(deg, gene_annotation, by = "gene_id", all.x = TRUE)

# Flag genes not in atlas as "unannotated"
deg_annotated$specificity[is.na(deg_annotated$specificity)] <- "unannotated"
deg_annotated$predominant_broad_organ[is.na(deg_annotated$predominant_broad_organ)] <- "unannotated"
deg_annotated$predominant_subtissue[is.na(deg_annotated$predominant_subtissue)] <- "unannotated"

# Add regulation direction
deg_annotated$regulation <- ifelse(deg_annotated$deg_flag == "yes",
                                    ifelse(deg_annotated$log2FoldChange > 0, "up", "down"),
                                    "non_DEG")

cat("\n=== Annotated DEG dataset ===\n")
cat("Total genes:", nrow(deg_annotated), "\n")
cat("DEGs:", sum(deg_annotated$deg_flag == "yes"), "\n")

cat("\n=== DEG specificity breakdown ===\n")
deg_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]
print(table(deg_only$specificity))

cat("\n=== DEG predominant broad organ ===\n")
print(table(deg_only$predominant_broad_organ, useNA = "ifany"))

cat("\n=== DEG predominant sub-tissue (top 15) ===\n")
print(sort(table(deg_only$predominant_subtissue, useNA = "ifany"), decreasing=TRUE)[1:15])

cat("\n=== DEG regulation by broad organ ===\n")
print(table(deg_only$predominant_broad_organ, deg_only$regulation))

# Save
saveRDS(deg_annotated, "/workspace/deg_annotated.rds")
cat("\n=== Saved annotated DEG data ===\n")


=== Tau index summary (sub-tissue level) ===
Genes with valid Tau: 20833 
Mean: 0.6622 
Median: 0.6611 
Min: 0.1265 
Max: 0.9983 
Quantiles:
    0%    25%    50%    75%   100% 
0.1265 0.5192 0.6611 0.8105 0.9983 

=== Tau index summary (broad-organ level) ===
Genes with valid Tau: 20833 
Mean: 0.4597 
Median: 0.4163 
Quantiles:
    0%    25%    50%    75%   100% 
0.0087 0.2702 0.4163 0.6140 0.9998 

=== Specificity classification ===

   constitutive tissue_specific 
           8061           12772 

=== Predominant broad organ distribution ===

      Flower   Leaf_Shoot         Root Seed_Silique     Seedling 
        7334         6965         1520         4773          241 

=== Predominant sub-tissue distribution (top 15) ===

        mature_pollen         seed_stage_10        senescing_leaf 
                 3580                  1659                  1616 
                 root shoot_apex_transition    inflorescence_apex 
                 1520                  1500                 

Export hierarchical tissue-specific CSV files

In [23]:

# ── Export tissue-specific CSV files in hierarchical structure ─────────────────
deg_annotated <- readRDS("/workspace/deg_annotated.rds")

# Base output directory
out_base <- "/mnt/results/tissue_specific_degs"
dir.create(out_base, showWarnings = FALSE, recursive = TRUE)

# Filter to DEGs only
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# Reorder columns for clarity
col_order <- c("gene_id", "baseMean", "log2FoldChange", "lfcSE", "stat", 
               "pvalue", "padj", "deg_flag", "regulation",
               "tau_subtissue", "tau_broad", "specificity",
               "predominant_broad_organ", "predominant_subtissue")
degs_only <- degs_only[, col_order]

cat("=== Exporting tissue-specific DEG files ===\n")
cat("Total DEGs to distribute:", nrow(degs_only), "\n\n")

# ── 1. Export per broad-organ / sub-tissue CSVs ────────────────────────────────
# Only for tissue-specific DEGs (skip constitutive and unannotated for tissue folders)
tissue_specific_degs <- degs_only[degs_only$specificity == "tissue_specific", ]

exported_files <- c()

for (bo in sort(unique(tissue_specific_degs$predominant_broad_organ))) {
  # Create broad organ directory
  organ_dir <- file.path(out_base, bo)
  dir.create(organ_dir, showWarnings = FALSE, recursive = TRUE)
  
  # Export all DEGs for this broad organ
  organ_degs <- tissue_specific_degs[tissue_specific_degs$predominant_broad_organ == bo, ]
  organ_file <- file.path(organ_dir, paste0(bo, "_all_DEGs.csv"))
  write.csv(organ_degs, organ_file, row.names = FALSE)
  exported_files <- c(exported_files, organ_file)
  cat(bo, ": ", nrow(organ_degs), " DEGs -> ", basename(organ_file), "\n", sep="")
  
  # Export per sub-tissue
  for (st in sort(unique(organ_degs$predominant_subtissue))) {
    st_degs <- organ_degs[organ_degs$predominant_subtissue == st, ]
    st_file <- file.path(organ_dir, paste0(st, "_DEGs.csv"))
    write.csv(st_degs, st_file, row.names = FALSE)
    exported_files <- c(exported_files, st_file)
    cat("  ", st, ": ", nrow(st_degs), " DEGs\n", sep="")
  }
}

# ── 2. Export constitutive DEGs ───────────────────────────────────────────────
constitutive_degs <- degs_only[degs_only$specificity == "constitutive", ]
const_file <- file.path(out_base, "constitutive_DEGs.csv")
write.csv(constitutive_degs, const_file, row.names = FALSE)
exported_files <- c(exported_files, const_file)
cat("\nConstitutive: ", nrow(constitutive_degs), " DEGs -> constitutive_DEGs.csv\n", sep="")

# ── 3. Export unannotated DEGs ─────────────────────────────────────────────────
unannotated_degs <- degs_only[degs_only$specificity == "unannotated", ]
unannot_file <- file.path(out_base, "unannotated_DEGs.csv")
write.csv(unannotated_degs, unannot_file, row.names = FALSE)
exported_files <- c(exported_files, unannot_file)
cat("Unannotated: ", nrow(unannotated_degs), " DEGs -> unannotated_DEGs.csv\n", sep="")

# ── 4. Export master annotated file (all genes, not just DEGs) ─────────────────
master_file <- file.path(out_base, "all_degs_with_tissue_annotation.csv")
write.csv(deg_annotated[, col_order], master_file, row.names = FALSE)
exported_files <- c(exported_files, master_file)
cat("\nMaster file (all genes): ", nrow(deg_annotated), " genes -> all_degs_with_tissue_annotation.csv\n", sep="")

cat("\n=== Total files exported:", length(exported_files), "===\n")
cat("\n=== File listing ===\n")
for (f in sort(exported_files)) {
  cat(gsub(out_base, "tissue_specific_degs", f), "\n")
}


=== Exporting tissue-specific DEG files ===
Total DEGs to distribute: 6942 

Flower: 1192 DEGs -> Flower_all_DEGs.csv
  carpel: 36 DEGs
  flower_stage_10_11: 21 DEGs
  flower_stage_12: 2 DEGs
  flower_stage_15: 10 DEGs
  flower_stage_9: 36 DEGs
  inflorescence_apex: 201 DEGs
  mature_pollen: 530 DEGs
  pedicel: 69 DEGs
  petal: 101 DEGs
  sepal: 120 DEGs
  stamen: 66 DEGs
Leaf_Shoot: 1399 DEGs -> Leaf_Shoot_all_DEGs.csv
  cauline_leaf: 77 DEGs
  cotyledon: 169 DEGs
  hypocotyl: 114 DEGs
  rosette_leaf: 38 DEGs
  rosette_vegetative: 129 DEGs
  senescing_leaf: 436 DEGs
  shoot_apex_transition: 212 DEGs
  shoot_apex_vegetative: 63 DEGs
  shoot_apex_vegetative_with_leaves: 59 DEGs
  stem: 102 DEGs
Root: 432 DEGs -> Root_all_DEGs.csv
  root: 432 DEGs
Seed_Silique: 609 DEGs -> Seed_Silique_all_DEGs.csv
  seed_stage_10: 180 DEGs
  seed_stage_6: 96 DEGs
  seed_stage_7: 62 DEGs
  seed_stage_8: 54 DEGs
  seed_stage_9: 116 DEGs
  silique_stage_3: 33 DEGs
  silique_stage_4: 49 DEGs
  silique_stage

Generate tissue DEG summary table

In [25]:

# ── Generate summary table: DEG counts per tissue with up/down breakdown ───────
deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# ── Sub-tissue level summary ───────────────────────────────────────────────────
build_summary <- function(df, group_col) {
  summary_list <- list()
  for (grp in sort(unique(df[[group_col]]))) {
    subset_df <- df[df[[group_col]] == grp, ]
    summary_list[[grp]] <- data.frame(
      tissue = grp,
      total_DEGs = nrow(subset_df),
      upregulated = sum(subset_df$regulation == "up"),
      downregulated = sum(subset_df$regulation == "down"),
      median_log2FC = round(median(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_log2FC = round(mean(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_tau_subtissue = round(mean(subset_df$tau_subtissue, na.rm = TRUE), 4),
      mean_tau_broad = round(mean(subset_df$tau_broad, na.rm = TRUE), 4),
      stringsAsFactors = FALSE
    )
  }
  do.call(rbind, summary_list)
}

# Sub-tissue summary
sub_summary <- build_summary(degs_only, "predominant_subtissue")
sub_summary <- sub_summary[order(-sub_summary$total_DEGs), ]

cat("=== Sub-tissue summary ===\n")
print(sub_summary)

# Broad-organ summary
broad_summary <- build_summary(degs_only, "predominant_broad_organ")
broad_summary <- broad_summary[order(-broad_summary$total_DEGs), ]

cat("\n=== Broad-organ summary ===\n")
print(broad_summary)

# ── Combined summary with hierarchical structure ───────────────────────────────
# Build a hierarchical summary: broad_organ -> sub_tissue
hierarchical_summary <- data.frame(
  broad_organ = character(),
  sub_tissue = character(),
  total_DEGs = integer(),
  upregulated = integer(),
  downregulated = integer(),
  median_log2FC = numeric(),
  mean_log2FC = numeric(),
  mean_tau_subtissue = numeric(),
  stringsAsFactors = FALSE
)

# Add tissue-specific DEGs grouped hierarchically
tissue_specific <- degs_only[degs_only$specificity == "tissue_specific", ]

for (bo in sort(unique(tissue_specific$predominant_broad_organ))) {
  for (st in sort(unique(tissue_specific$predominant_subtissue[tissue_specific$predominant_broad_organ == bo]))) {
    subset_df <- tissue_specific[tissue_specific$predominant_broad_organ == bo & 
                                  tissue_specific$predominant_subtissue == st, ]
    hierarchical_summary <- rbind(hierarchical_summary, data.frame(
      broad_organ = bo,
      sub_tissue = st,
      total_DEGs = nrow(subset_df),
      upregulated = sum(subset_df$regulation == "up"),
      downregulated = sum(subset_df$regulation == "down"),
      median_log2FC = round(median(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_log2FC = round(mean(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_tau_subtissue = round(mean(subset_df$tau_subtissue, na.rm = TRUE), 4),
      stringsAsFactors = FALSE
    ))
  }
}

# Add constitutive and unannotated rows
for (spec in c("constitutive", "unannotated")) {
  subset_df <- degs_only[degs_only$specificity == spec, ]
  hierarchical_summary <- rbind(hierarchical_summary, data.frame(
    broad_organ = spec,
    sub_tissue = "(all)",
    total_DEGs = nrow(subset_df),
    upregulated = sum(subset_df$regulation == "up"),
    downregulated = sum(subset_df$regulation == "down"),
    median_log2FC = round(median(subset_df$log2FoldChange, na.rm = TRUE), 4),
    mean_log2FC = round(mean(subset_df$log2FoldChange, na.rm = TRUE), 4),
    mean_tau_subtissue = round(mean(subset_df$tau_subtissue, na.rm = TRUE), 4),
    stringsAsFactors = FALSE
  ))
}

# Add total row
hierarchical_summary <- rbind(hierarchical_summary, data.frame(
  broad_organ = "TOTAL",
  sub_tissue = "(all)",
  total_DEGs = nrow(degs_only),
  upregulated = sum(degs_only$regulation == "up"),
  downregulated = sum(degs_only$regulation == "down"),
  median_log2FC = round(median(degs_only$log2FoldChange, na.rm = TRUE), 4),
  mean_log2FC = round(mean(degs_only$log2FoldChange, na.rm = TRUE), 4),
  mean_tau_subtissue = round(mean(degs_only$tau_subtissue, na.rm = TRUE), 4),
  stringsAsFactors = FALSE
))

cat("\n=== Hierarchical summary ===\n")
print(hierarchical_summary)

# Export
write.csv(hierarchical_summary, "/mnt/results/tissue_deg_summary.csv", row.names = FALSE)
cat("\n=== Summary saved to /mnt/results/tissue_deg_summary.csv ===\n")


=== Sub-tissue summary ===
                                                             tissue total_DEGs
unannotated                                             unannotated       1123
senescing_leaf                                       senescing_leaf        665
mature_pollen                                         mature_pollen        636
root                                                           root        517
shoot_apex_transition                         shoot_apex_transition        414
inflorescence_apex                               inflorescence_apex        397
seed_stage_10                                         seed_stage_10        320
shoot_apex_vegetative                         shoot_apex_vegetative        258
cotyledon                                                 cotyledon        234
rosette_vegetative                               rosette_vegetative        226
seed_stage_9                                           seed_stage_9        183
seed_stage_6             

Generate bar chart of DEG counts by organ

In [27]:

# ── Visualization 1: Bar chart of DEG counts per broad organ (stacked up/down) ─
suppressPackageStartupMessages({
  library(ggplot2)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# Prepare data — exclude unannotated for the tissue plot
tissue_degs <- degs_only[degs_only$specificity == "tissue_specific", ]

# Order broad organs by total DEG count
organ_order <- names(sort(table(tissue_degs$predominant_broad_organ), decreasing = TRUE))
tissue_degs$predominant_broad_organ <- factor(tissue_degs$predominant_broad_organ, levels = organ_order)

# Color palette (colorblind-friendly)
up_color <- "#0072B2"   # blue
down_color <- "#D55E00" # vermillion

p1 <- ggplot(tissue_degs, aes(x = predominant_broad_organ, fill = regulation)) +
  geom_bar(position = "stack", width = 0.7) +
  scale_fill_manual(values = c("up" = up_color, "down" = down_color),
                    labels = c("up" = "Upregulated", "down" = "Downregulated"),
                    name = "Regulation") +
  labs(
    title = "Radiation-Responsive DEGs by Tissue (Broad Organ)",
    subtitle = "Arabidopsis thaliana — OSD-498/510 radiation effect",
    x = "Broad Organ",
    y = "Number of DEGs"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    text = element_text(family = "Liberation Sans"),
    plot.title = element_text(face = "bold", size = 16),
    plot.subtitle = element_text(size = 11, color = "grey40"),
    axis.text.x = element_text(angle = 30, hjust = 1, size = 12),
    legend.position = "top",
    panel.grid.minor = element_blank()
  ) +
  geom_text(stat = "count", aes(label = after_stat(count)), 
            position = position_stack(vjust = 0.5), size = 3.5, color = "white")

ggsave("/mnt/results/fig1_deg_counts_by_organ.png", p1, width = 8, height = 6, dpi = 300, bg = "white")
cat("Saved fig1_deg_counts_by_organ.png\n")


Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Saved fig1_deg_counts_by_organ.png


Generate volcano plots per organ

In [29]:

# ── Visualization 2: Volcano plots per broad organ (multi-panel) ───────────────
suppressPackageStartupMessages({
  library(ggplot2)
  library(ggrepel)
})

# Prepare data — tissue-specific DEGs only, exclude unannotated
tissue_degs <- degs_only[degs_only$specificity == "tissue_specific", ]

# For volcano plots, we need all genes (not just DEGs) colored by tissue assignment
# Use the full annotated dataset
all_genes <- deg_annotated
all_genes$neg_log10_padj <- -log10(all_genes$padj)

# For each broad organ, create a volcano plot
organs <- sort(unique(tissue_degs$predominant_broad_organ))

# Create a combined plot with one panel per organ
volcano_list <- list()

for (bo in organs) {
  # Genes assigned to this organ
  organ_genes <- all_genes[all_genes$predominant_broad_organ == bo, ]
  
  # Also include constitutive genes as grey background
  constitutive <- all_genes[all_genes$specificity == "constitutive", ]
  
  # Combine: constitutive as background, organ-specific as colored
  plot_data <- rbind(
    data.frame(constitutive[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")], 
               category = "constitutive"),
    data.frame(organ_genes[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")],
               category = bo)
  )
  
  # Cap neg_log10_padj for visualization
  plot_data$neg_log10_padj[plot_data$neg_log10_padj > 50] <- 50
  
  p <- ggplot(plot_data, aes(x = log2FoldChange, y = neg_log10_padj, color = category)) +
    geom_point(data = subset(plot_data, category == "constitutive"), 
               color = "grey80", alpha = 0.3, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "no"),
               color = "grey60", alpha = 0.4, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "up"),
               color = up_color, alpha = 0.7, size = 1.2) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "down"),
               color = down_color, alpha = 0.7, size = 1.2) +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.4) +
    geom_vline(xintercept = 0, linetype = "dotted", color = "grey50", linewidth = 0.4) +
    labs(
      title = bo,
      x = "log2 Fold Change",
      y = "-log10(padj)"
    ) +
    theme_minimal(base_size = 11) +
    theme(
      text = element_text(family = "Liberation Sans"),
      plot.title = element_text(face = "bold", size = 12, hjust = 0.5),
      legend.position = "none",
      panel.grid.minor = element_blank()
    ) +
    coord_cartesian(xlim = c(-4, 7), ylim = c(0, 52))
  
  volcano_list[[bo]] <- p
}

# Combine using patchwork-like approach with cowplot or gridExtra
# Use gridExtra
suppressPackageStartupMessages(library(gridExtra))

# Create a legend plot
legend_data <- data.frame(
  x = c(1, 2, 3),
  label = c("Upregulated DEG", "Downregulated DEG", "Non-DEG / Constitutive"),
  color = c(up_color, down_color, "grey70")
)

combined_volcano <- do.call(grid.arrange, c(volcano_list, ncol = 3, 
  top = textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes", 
                 gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))))

ggsave("/mnt/results/fig2_volcano_plots_by_organ.png", 
       arrangeGrob(grobs = volcano_list, ncol = 3,
         top = textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes",
                        gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))),
       width = 14, height = 10, dpi = 300, bg = "white")
cat("Saved fig2_volcano_plots_by_organ.png\n")


Warning message:
“package ‘ggrepel’ was built under R version 4.4.3”
Warning message:
“package ‘gridExtra’ was built under R version 4.4.3”
Error in h(simpleError(msg, call)): error in evaluating the argument 'args' in selecting a method for function 'do.call': could not find function "textGrob"
Traceback:

1. .handleSimpleError(function (cond) 
 . .Internal(C_tryCatchHelper(addr, 1L, cond)), "could not find function \"textGrob\"", 
 .     base::quote(textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes", 
 .         gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))))
2. h(simpleError(msg, call))
3. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "error in evaluating the argument 'args' in selecting a method for function 'do.call

Regenerate volcano plots with grid loaded

In [31]:

# ── Fix: load grid package and regenerate volcano plots ────────────────────────
suppressPackageStartupMessages({
  library(grid)
  library(gridExtra)
  library(ggplot2)
})

# Rebuild volcano plots (variables still in scope from previous cell)
# Rebuild plot_data for each organ
volcano_list <- list()

for (bo in organs) {
  organ_genes <- all_genes[all_genes$predominant_broad_organ == bo, ]
  
  plot_data <- rbind(
    data.frame(constitutive[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")],
               category = "constitutive"),
    data.frame(organ_genes[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")],
               category = bo)
  )
  
  plot_data$neg_log10_padj[plot_data$neg_log10_padj > 50] <- 50
  
  p <- ggplot(plot_data, aes(x = log2FoldChange, y = neg_log10_padj)) +
    geom_point(data = subset(plot_data, category == "constitutive"),
               color = "grey80", alpha = 0.3, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "no"),
               color = "grey60", alpha = 0.4, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "up"),
               color = up_color, alpha = 0.7, size = 1.2) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "down"),
               color = down_color, alpha = 0.7, size = 1.2) +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.4) +
    geom_vline(xintercept = 0, linetype = "dotted", color = "grey50", linewidth = 0.4) +
    labs(
      title = bo,
      x = "log2 Fold Change",
      y = "-log10(padj)"
    ) +
    theme_minimal(base_size = 11) +
    theme(
      text = element_text(family = "Liberation Sans"),
      plot.title = element_text(face = "bold", size = 12, hjust = 0.5),
      legend.position = "none",
      panel.grid.minor = element_blank()
    ) +
    coord_cartesian(xlim = c(-4, 7), ylim = c(0, 52))
  
  volcano_list[[bo]] <- p
}

# Save with grid textGrob for title
ggsave("/mnt/results/fig2_volcano_plots_by_organ.png",
       arrangeGrob(grobs = volcano_list, ncol = 3,
         top = textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes",
                        gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))),
       width = 14, height = 10, dpi = 300, bg = "white")
cat("Saved fig2_volcano_plots_by_organ.png\n")


Saved fig2_volcano_plots_by_organ.png


Generate heatmap of top DEGs by tissue

In [33]:

# ── Visualization 3: Heatmap of top DEGs x tissues (expression specificity) ───
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
})

# Load atlas
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
sub_tissue_expr <- atlas$sub_tissue_expr
deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# Select top 50 DEGs by padj (most significant)
top_degs <- degs_only[order(degs_only$padj), ][1:50, ]

# Get their expression in the atlas (sub-tissue level)
top_genes <- top_degs$gene_id
genes_in_atlas <- intersect(top_genes, rownames(sub_tissue_expr))

# Expression matrix for these genes
expr_top <- sub_tissue_expr[genes_in_atlas, , drop = FALSE]

# Z-score normalize per gene (row)
expr_z <- t(scale(t(expr_top)))

# Order columns by broad organ for visual grouping
sub_to_broad <- unique(atlas$all_samples[, c("sub_tissue", "broad_organ")])
col_order <- sub_to_broad[order(sub_to_broad$broad_organ), "sub_tissue"]
col_order <- intersect(col_order, colnames(expr_z))
expr_z <- expr_z[, col_order]

# Create column annotation (broad organ)
col_anno_data <- data.frame(
  broad_organ = sub_to_broad$broad_organ[match(col_order, sub_to_broad$sub_tissue)],
  row.names = col_order
)

# Color palette for broad organs
organ_colors <- c(
  "Root" = "#0072B2",
  "Seedling" = "#009E73",
  "Leaf_Shoot" = "#E69F00",
  "Flower" = "#CC79A7",
  "Seed_Silique" = "#D55E00"
)

col_anno <- HeatmapAnnotation(
  Broad_Organ = col_anno_data$broad_organ,
  col = list(Broad_Organ = organ_colors),
  annotation_name_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"),
  annotation_legend_param = list(title_gp = gpar(fontsize = 10, fontfamily = "Liberation Sans"),
                                  labels_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"))
)

# Row annotation: regulation direction
row_anno_data <- data.frame(
  regulation = top_degs$regulation[match(genes_in_atlas, top_degs$gene_id)],
  row.names = genes_in_atlas
)

row_anno <- rowAnnotation(
  Regulation = row_anno_data$regulation,
  col = list(Regulation = c("up" = up_color, "down" = down_color)),
  annotation_name_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"),
  annotation_legend_param = list(title_gp = gpar(fontsize = 10, fontfamily = "Liberation Sans"),
                                  labels_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"))
)

# Create heatmap
png("/mnt/results/fig3_heatmap_top_degs.png", width = 12, height = 8, units = "in", res = 300, bg = "white")

ht <- Heatmap(expr_z,
  name = "Z-score\nExpression",
  col = colorRamp2(c(-2, 0, 2), c(down_color, "white", up_color)),
  top_annotation = col_anno,
  right_annotation = row_anno,
  cluster_columns = FALSE,
  cluster_rows = TRUE,
  show_row_names = TRUE,
  show_column_names = TRUE,
  row_names_gp = gpar(fontsize = 7, fontfamily = "Liberation Sans"),
  column_names_gp = gpar(fontsize = 7, fontfamily = "Liberation Sans"),
  column_names_rot = 45,
  column_title = "Top 50 DEGs by Significance — Tissue Expression Specificity",
  column_title_gp = gpar(fontface = "bold", fontsize = 13, fontfamily = "Liberation Sans"),
  heatmap_legend_param = list(title_gp = gpar(fontsize = 10, fontfamily = "Liberation Sans"),
                               labels_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"))
)

draw(ht)
dev.off()
cat("Saved fig3_heatmap_top_degs.png\n")


Warning message:
“package ‘circlize’ was built under R version 4.4.3”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in

Prepare data for UpSet diagram

In [35]:

# ── Visualization 4: UpSet diagram of DEG overlaps across broad organs ─────────
suppressPackageStartupMessages({
  library(UpSetR)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# For UpSet: create binary matrix of gene x broad_organ membership
# Only tissue-specific DEGs (each gene belongs to exactly one organ in our assignment)
# But we can also show constitutive + unannotated as separate sets

# Actually, since each tissue-specific gene is assigned to ONE organ, 
# the overlaps would be empty. Let's instead show the overlap at the GENE level:
# which genes are DEGs AND tissue-specific to each organ?

# Better approach: show the overlap of DEG sets across organs using ALL DEGs
# that have expression in each organ above a threshold.
# 
# Alternative: show the overlap between tissue-specific, constitutive, and unannotated categories
# That's not very informative either.
#
# Best approach for UpSet: For each broad organ, define the set of DEGs whose 
# atlas expression is above median in that organ (i.e., "expressed" in that organ).
# This shows which DEGs are active in multiple organs.

atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
broad_organ_expr <- atlas$broad_organ_expr

# Get DEGs that are in the atlas
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]

# Fix: typo above, redo properly
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]


Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found
Traceback:

1. `[.data.frame`(degs_only, deggs_only$gene_id %in% rownames(broad_organ_expr), 
 .     )
2. deggs_only$gene_id %in% rownames(broad_organ_expr)
3. .handleSimpleError(function (cond) 
 . .Internal(C_tryCatchHelper(addr, 1L, cond)), "object 'deggs_only' not found", 
 .     base::quote(eval(expr, envir)))
4. h(simpleError(msg, call))
5. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found", 
 .     base::quote(h(simpleError(msg, call))))

Prepare UpSet data attempt 2

In [37]:

# ── UpSet diagram: DEG overlaps across broad organs ────────────────────────────
suppressPackageStartupMessages({
  library(UpSetR)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
broad_organ_expr <- atlas$broad_organ_expr

# Get DEGs that are in the atlas
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]


Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found
Traceback:

1. `[.data.frame`(degs_only, deggs_only$gene_id %in% rownames(broad_organ_expr), 
 .     )
2. deggs_only$gene_id %in% rownames(broad_organ_expr)
3. .handleSimpleError(function (cond) 
 . .Internal(C_tryCatchHelper(addr, 1L, cond)), "object 'deggs_only' not found", 
 .     base::quote(eval(expr, envir)))
4. h(simpleError(msg, call))
5. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found", 
 .     base::quote(h(simpleError(msg, call))))

Generate UpSet diagram of organ overlaps

In [39]:

# ── UpSet diagram: DEG overlaps across broad organs ────────────────────────────
suppressPackageStartupMessages({
  library(UpSetR)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
broad_organ_expr <- atlas$broad_organ_expr

# Get DEGs that are in the atlas
degs_in_atlas <- degs_only[degs_only$gene_id %in% rownames(broad_organ_expr), ]
cat("DEGs in atlas:", nrow(degs_in_atlas), "\n")

# For each broad organ, define the set of DEGs "expressed" in that organ
# (expression above the gene's median across all organs — i.e., relatively high)
# This shows which DEGs are active in multiple organs

# Get expression for DEGs
expr_degs <- broad_organ_expr[degs_in_atlas$gene_id, , drop = FALSE]
cat("Expression matrix for DEGs:", nrow(expr_degs), "x", ncol(expr_degs), "\n")

# For each gene, determine which organs it's "expressed" in (above gene median)
gene_medians <- apply(expr_degs, 1, median, na.rm = TRUE)

# Build binary matrix: gene x organ (1 = expressed above median, 0 = not)
binary_matrix <- matrix(0, nrow = nrow(expr_degs), ncol = ncol(expr_degs))
rownames(binary_matrix) <- rownames(expr_degs)
colnames(binary_matrix) <- colnames(expr_degs)

for (i in seq_len(nrow(expr_degs))) {
  binary_matrix[i, ] <- as.integer(expr_degs[i, ] > gene_medians[i])
}

# Convert to data frame for UpSetR
binary_df <- as.data.frame(binary_matrix)
binary_df$gene_id <- rownames(binary_df)

cat("\n=== Organ set sizes (DEGs expressed above median) ===\n")
print(colSums(binary_matrix))

# Create UpSet plot
png("/mnt/results/fig4_upset_organ_overlaps.png", width = 10, height = 6, units = "in", res = 300, bg = "white")

upset_plot <- upset(binary_df, 
  sets = colnames(expr_degs),
  order.by = "freq",
  nsets = 5,
  nintersects = 20,
  sets.bar.color = c("#0072B2", "#009E73", "#E69F00", "#CC79A7", "#D55E00"),
  main.bar.color = "grey30",
  matrix.color = "#0279EE",
  point.size = 3,
  line.size = 0.8,
  text.scale = 1.2,
  mb.ratio = c(0.6, 0.4),
  mainbar.y.label = "DEG Intersection Size",
  sets.x.label = "DEGs Expressed Above Median",
  labels = c("Intersection", "Set")
)

print(upset_plot)
dev.off()
cat("\nSaved fig4_upset_organ_overlaps.png\n")


DEGs in atlas: 5819 
Expression matrix for DEGs: 5819 x 5 

=== Organ set sizes (DEGs expressed above median) ===
    Seedling   Leaf_Shoot         Root       Flower Seed_Silique 
        2041         2178         2717         2629         2073 
Error in upset(binary_df, sets = colnames(expr_degs), order.by = "freq", : unused argument (labels = c("Intersection", "Set"))
Traceback:


Generate UpSet diagram fixed

In [41]:

# ── Fix UpSet diagram (remove invalid 'labels' argument) ───────────────────────
png("/mnt/results/fig4_upset_organ_overlaps.png", width = 10, height = 6, units = "in", res = 300, bg = "white")

upset_plot <- upset(binary_df, 
  sets = colnames(expr_degs),
  order.by = "freq",
  nsets = 5,
  nintersects = 20,
  sets.bar.color = c("#0072B2", "#009E73", "#E69F00", "#CC79A7", "#D55E00"),
  main.bar.color = "grey30",
  matrix.color = "#0279EE",
  point.size = 3,
  line.size = 0.8,
  text.scale = 1.2,
  mb.ratio = c(0.6, 0.4),
  mainbar.y.label = "DEG Intersection Size",
  sets.x.label = "DEGs Expressed Above Median"
)

print(upset_plot)
dev.off()
cat("Saved fig4_upset_organ_overlaps.png\n")


Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.
ℹ The deprecated feature was likely used in the UpSetR package.
  Please report the issue at <https://github.com/hms-dbmi/UpSetR/issues>.”
Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.
ℹ The deprecated feature was likely used in the UpSetR package.
  Please report the issue at <https://github.com/hms-dbmi/UpSetR/issues>.”
Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.
ℹ The deprecated feature was likely used in the UpSetR package.
  Please report the issue at <https://github.com/hms-dbmi/UpSetR/issues>.”
Saved fig4_upset_organ_overlaps.png


Set up PDF generation infrastructure

In [43]:

# ── Generate the PDF instruction manual ────────────────────────────────────────
# Mixed-level educational document using ReportLab (per pdf-report-generation skill)

import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                 PageBreak, HRFlowable, KeepTogether, Image)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from reportlab.lib.units import inch

# ── Phylo Brand Colors ──
PHYLO_BLACK     = HexColor("#000000")
PHYLO_WARM_GRAY = HexColor("#ECE9E2")
PHYLO_OFF_WHITE = HexColor("#FAF9F3")
PHYLO_GOLD      = HexColor("#D4A04A")
HEADING_COLOR   = HexColor("#111111")
BODY_TEXT       = HexColor("#2C2A26")
MUTED_TEXT      = HexColor("#8A8378")
TABLE_HEADER_BG = PHYLO_GOLD
TABLE_HEADER_FG = HexColor("#FFFFFF")
TABLE_ALT_ROW   = HexColor("#F9F7F3")
TABLE_BORDER    = HexColor("#D5CFC5")
DIVIDER_COLOR   = PHYLO_GOLD
CALLOUT_BG      = PHYLO_OFF_WHITE
CALLOUT_BORDER  = PHYLO_GOLD

# ── Fonts ──
FONT_HEADING = "Helvetica-Bold"
FONT_BODY    = "Helvetica"
FONT_ITALIC  = "Helvetica-Oblique"
FONT_MONO    = "Courier"

# ── Styles ──
styles = getSampleStyleSheet()

styles.add(ParagraphStyle(name="ReportTitle", fontName=FONT_HEADING, fontSize=26,
    textColor=HEADING_COLOR, spaceBefore=0, spaceAfter=6, leading=32))
styles.add(ParagraphStyle(name="Subtitle", fontName=FONT_BODY, fontSize=12,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="Attribution", fontName=FONT_ITALIC, fontSize=10,
    textColor=MUTED_TEXT, spaceAfter=8))
styles.add(ParagraphStyle(name="SectionHead", fontName=FONT_HEADING, fontSize=18,
    textColor=HEADING_COLOR, spaceBefore=24, spaceAfter=10))
styles.add(ParagraphStyle(name="SubHead", fontName=FONT_HEADING, fontSize=13,
    textColor=HEADING_COLOR, spaceBefore=16, spaceAfter=6))
styles.add(ParagraphStyle(name="Body", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_JUSTIFY, spaceAfter=8, leading=15))
styles.add(ParagraphStyle(name="Caption", fontName=FONT_ITALIC, fontSize=9,
    textColor=MUTED_TEXT, alignment=TA_CENTER, spaceAfter=14))
styles.add(ParagraphStyle(name="CalloutText", fontName=FONT_BODY, fontSize=10,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=14))
styles.add(ParagraphStyle(name="CalloutTitle", fontName=FONT_HEADING, fontSize=10.5,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="CodeStyle", fontName=FONT_MONO, fontSize=8.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=11,
    leftIndent=12, rightIndent=12))
styles.add(ParagraphStyle(name="BulletStyle", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=15,
    leftIndent=20, bulletIndent=8))
styles.add(ParagraphStyle(name="TableCell", fontName=FONT_BODY, fontSize=9,
    textColor=BODY_TEXT, alignment=TA_LEFT, leading=12))
styles.add(ParagraphStyle(name="TableHeader", fontName=FONT_HEADING, fontSize=9,
    textColor=TABLE_HEADER_FG, alignment=TA_LEFT, leading=12))

# ── Helper functions ──
def divider(width=480):
    return HRFlowable(width=width, thickness=1, color=DIVIDER_COLOR,
                      spaceAfter=10, spaceBefore=4)

def callout_box(title, text, width=460):
    """Beginner-friendly sidebar callout with gold left border."""
    content = []
    if title:
        content.append(Paragraph(title, styles["CalloutTitle"]))
    content.append(Paragraph(text, styles["CalloutText"]))
    t = Table([[content]], colWidths=[width])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), CALLOUT_BG),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("LINEBEFORE", (0, 0), (0, -1), 3, PHYLO_GOLD),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
        ("LEFTPADDING", (0, 0), (-1, -1), 14),
        ("RIGHTPADDING", (0, 0), (-1, -1), 14),
    ]))
    t.hAlign = "CENTER"
    return t

def code_block(code_text, width=460):
    """Monospace code block with light background."""
    # Escape XML special chars
    escaped = code_text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    # Preserve line breaks
    escaped = escaped.replace("\n", "<br/>")
    p = Paragraph(escaped, styles["CodeStyle"])
    t = Table([[p]], colWidths=[width])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), HexColor("#F5F3EE")),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
        ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
    ]))
    t.hAlign = "CENTER"
    return t

def make_table(headers, rows, col_widths):
    """Standard Phylo-styled table."""
    data = [[Paragraph(f'<b>{h}</b>', styles["TableHeader"]) for h in headers]]
    for row in rows:
        data.append([Paragraph(str(c), styles["TableCell"]) for c in row])
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), TABLE_HEADER_BG),
        ("TEXTCOLOR", (0, 0), (-1, 0), TABLE_HEADER_FG),
        *[("BACKGROUND", (0, i), (-1, i), TABLE_ALT_ROW) for i in range(2, len(data), 2)],
        ("GRID", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("BOX", (0, 0), (-1, -1), 0.75, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    t.hAlign = "CENTER"
    return t

# ── Page header/footer ──
def page_header_footer(canvas, doc):
    canvas.saveState()
    w, h = letter
    canvas.setFont("Helvetica", 9)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawString(60, h - 40, "Tissue-Specific DEG Analysis: Instruction Manual")
    canvas.setStrokeColor(PHYLO_GOLD)
    canvas.setLineWidth(1)
    canvas.line(60, h - 48, w - 60, h - 48)
    canvas.setStrokeColor(TABLE_BORDER)
    canvas.setLineWidth(0.75)
    canvas.line(60, 40, w - 60, 40)
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawCentredString(w / 2, 26, f"Page {doc.page}")
    canvas.restoreState()

# ── Build the document ──
output_path = "/mnt/results/instruction_manual_tissue_specific_deg.pdf"
doc = SimpleDocTemplate(output_path, pagesize=letter,
                        topMargin=52, bottomMargin=52,
                        leftMargin=60, rightMargin=60)
story = []

print("Building PDF manual...")


Building PDF manual...


Add title page and sections 1-3 to PDF

In [45]:

# ── PAGE 1: Title Page ─────────────────────────────────────────────────────────
story.append(Spacer(1, 40))
story.append(Paragraph("Tissue-Specific Separation of Radiation-Responsive DEGs", styles["ReportTitle"]))
story.append(Paragraph("An Educational Instruction Manual for R-Based Transcriptomic Analysis", styles["Subtitle"]))
story.append(Spacer(1, 8))
story.append(Paragraph("<i>Generated by Biomni (Phylo)  |  2026-07-09</i>", styles["Attribution"]))
story.append(Spacer(1, 24))

# Brief abstract
story.append(Paragraph(
    "This manual accompanies an R script that separates differentially expressed genes (DEGs) "
    "from an Arabidopsis thaliana radiation experiment into tissue-specific response groups. "
    "The dataset comes from NASA GeneLab studies OSD-498 and OSD-510, which examined the "
    "transcriptional effects of ionizing radiation on whole Arabidopsis seedlings. Because the "
    "original experiment used whole seedlings (not separated tissues), we use the AtGenExpress "
    "developmental expression atlas to infer which tissues each radiation-responsive gene is "
    "normally most active in, then group DEGs accordingly.", styles["Body"]))

story.append(divider())

# ── Table of Contents ──
story.append(Paragraph("Contents", styles["SubHead"]))
toc_items = [
    "1.  Introduction: The Experiment and the Data",
    "2.  Background: Differential Expression and DESeq2",
    "3.  The Tissue-Specificity Concept: Expression Atlases and the Tau Index",
    "4.  Code Walkthrough: Section-by-Section Explanation",
    "5.  Understanding the Outputs: CSVs, Summary Table, and Figures",
    "6.  How to Run the Code: Prerequisites and Execution",
    "7.  Interpretation Guide: What Tissue-Specific Radiation Responses Mean",
    "8.  Exercises and Extensions for Further Exploration",
]
for item in toc_items:
    story.append(Paragraph(item, styles["BulletStyle"]))

story.append(PageBreak())

# ── SECTION 1: Introduction ────────────────────────────────────────────────────
story.append(Paragraph("1. Introduction: The Experiment and the Data", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("1.1 The Biological Question", styles["SubHead"]))
story.append(Paragraph(
    "When plants are exposed to ionizing radiation (such as gamma rays or cosmic radiation), "
    "their cells activate DNA damage response pathways, alter cell cycle progression, and "
    "modulate stress-responsive gene expression. But different plant tissues and organs may "
    "respond differently to radiation. Root cells, which are underground and normally shielded, "
    "might mount a different transcriptional response than leaf cells, which are adapted to "
    "handle light and oxidative stress. Understanding tissue-specific radiation responses is "
    "important for space agriculture, where plants will grow in high-radiation environments "
    "beyond Earth's protective magnetic field.", styles["Body"]))

story.append(Paragraph("1.2 The Dataset", styles["SubHead"]))
story.append(Paragraph(
    "The input file <font face='Courier'>DEG_OSD498_510_radiation_effect.csv</font> contains "
    "the results of a differential expression analysis performed with DESeq2, a widely used "
    "R package for analyzing RNA-seq count data. The data comes from NASA GeneLab studies "
    "OSD-498 and OSD-510, which profiled the transcriptome of Arabidopsis thaliana seedlings "
    "exposed to ionizing radiation.", styles["Body"]))

story.append(Paragraph("The file contains 23,573 genes and 8 columns:", styles["Body"]))

deg_table = make_table(
    ["Column", "Description", "What It Tells You"],
    [
        ["gene_id", "AGI locus identifier (e.g., AT5G60250)", "Unique gene name in Arabidopsis"],
        ["baseMean", "Average normalized expression across all samples", "How highly expressed the gene is overall"],
        ["log2FoldChange", "Log2 ratio of expression (irradiated / control)", "Direction and magnitude of change; positive = up, negative = down"],
        ["lfcSE", "Standard error of the log2 fold change", "Uncertainty in the fold change estimate"],
        ["stat", "Wald test statistic", "Used to compute the p-value"],
        ["pvalue", "Raw p-value from the Wald test", "Statistical significance before correction"],
        ["padj", "Adjusted p-value (Benjamini-Hochberg FDR)", "Significance after multiple testing correction"],
        ["deg_flag", "yes / no flag", "Whether the gene is a differentially expressed gene (DEG)"],
    ],
    [90, 160, 210]
)
story.append(deg_table)
story.append(Spacer(1, 8))

story.append(Paragraph(
    "Of the 23,573 genes, <b>6,942 are flagged as DEGs</b> (deg_flag = 'yes'). These are the "
    "genes whose expression significantly changed in response to radiation.", styles["Body"]))

story.append(callout_box(
    "BEGINNER'S CORNER: What is Arabidopsis thaliana?",
    "Arabidopsis thaliana is a small flowering plant widely used as a model organism in plant "
    "biology, much like E. coli is used in microbiology or Drosophila in genetics. It has a "
    "small genome (~135 megabases, ~27,000 genes), a short life cycle (~6 weeks), and is easy "
    "to grow in the lab. Every gene in Arabidopsis has a unique identifier called an AGI code "
    "(Arabidopsis Genome Initiative), formatted as AT{chromosome}G{number}, e.g., AT5G60250 "
    "means chromosome 5, gene 60250."
))

story.append(callout_box(
    "BEGINNER'S CORNER: What is ionizing radiation?",
    "Ionizing radiation carries enough energy to damage DNA directly, causing single- and "
    "double-strand breaks. In space, astronauts and plants are exposed to galactic cosmic "
    "rays (high-energy protons and heavy ions) and solar particle events. On Earth, gamma "
    "rays from radioactive sources (like Cobalt-60) are used to simulate some aspects of "
    "space radiation in lab experiments. Plants have evolved DNA damage response pathways "
    "(involving proteins like ATM, ATR, and SOG1) that detect damage and activate repair genes."
))

story.append(PageBreak())

# ── SECTION 2: Background ──────────────────────────────────────────────────────
story.append(Paragraph("2. Background: Differential Expression and DESeq2", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("2.1 What is Differential Expression?", styles["SubHead"]))
story.append(Paragraph(
    "Differential expression analysis compares gene expression levels between two or more "
    "conditions (e.g., irradiated vs. control plants) to identify genes that are significantly "
    "upregulated (expressed more) or downregulated (expressed less) in one condition relative "
    "to the other. The output is a table with a fold change and a p-value for every gene.", styles["Body"]))

story.append(Paragraph("2.2 Key Concepts in the DESeq2 Output", styles["SubHead"]))

story.append(Paragraph("<b>Log2 Fold Change (log2FC):</b> This is the most important column. "
    "It tells you the direction and magnitude of expression change. A log2FC of +1 means the "
    "gene's expression doubled; -1 means it halved; +2 means it quadrupled. We use log2 "
    "(rather than raw fold change) because it makes up- and down-regulation symmetric: "
    "+2 and -2 represent the same magnitude of change in opposite directions.", styles["Body"]))

story.append(Paragraph("<b>Adjusted p-value (padj):</b> When testing thousands of genes "
    "simultaneously, some will appear significant by chance alone. The adjusted p-value "
    "(using the Benjamini-Hochberg procedure) controls the false discovery rate (FDR). "
    "A common threshold is padj &lt; 0.05, meaning we expect at most 5% of the genes we "
    "call significant to be false positives.", styles["Body"]))

story.append(callout_box(
    "BEGINNER'S CORNER: Why log2 instead of raw fold change?",
    "If a gene goes from 100 to 200 reads, that's a 2-fold increase. If it goes from 100 to 50, "
    "that's a 0.5-fold change. On a linear scale, 'doubling' (2x) and 'halving' (0.5x) look "
    "very different in magnitude. But on a log2 scale: log2(2) = +1 and log2(0.5) = -1. "
    "Now they're symmetric! This makes it much easier to visualize and compare up- and "
    "down-regulation on the same plot (like a volcano plot)."
))

story.append(Paragraph("2.3 The Challenge: No Tissue Information", styles["SubHead"]))
story.append(Paragraph(
    "The critical challenge with this dataset is that it comes from <b>whole seedling</b> "
    "experiments. The researchers harvested entire Arabidopsis seedlings (roots, leaves, "
    "stems, and all) and extracted RNA from the pooled tissue. This means the DEG file "
    "contains no tissue column — we cannot directly tell whether a radiation-responsive gene "
    "is primarily active in roots, leaves, flowers, or elsewhere.", styles["Body"]))

story.append(Paragraph(
    "To address this, we use an <b>expression atlas</b> approach: we look up where each gene "
    "is normally most highly expressed (across many tissues, measured in a separate reference "
    "dataset), and use that information to infer which tissue's biology the gene is most "
    "relevant to. This is an inference, not a direct measurement, but it is a well-established "
    "approach in plant genomics.", styles["Body"]))

story.append(PageBreak())

# ── SECTION 3: Tissue-Specificity Concept ──────────────────────────────────────
story.append(Paragraph("3. The Tissue-Specificity Concept: Expression Atlases and the Tau Index", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("3.1 The AtGenExpress Developmental Atlas", styles["SubHead"]))
story.append(Paragraph(
    "The AtGenExpress project (Schmid et al. 2005, Nature Genetics) is the canonical reference "
    "for Arabidopsis tissue-specific gene expression. It profiled gene expression across 79+ "
    "diverse tissue samples covering the entire plant life cycle, from seeds to senescing "
    "leaves, using the Affymetrix ATH1 microarray. The data is publicly available from the "
    "Gene Expression Omnibus (GEO) under accession numbers GSE5629 through GSE5634.", styles["Body"]))

story.append(Paragraph(
    "We download six GEO series, each covering a different organ system:", styles["Body"]))

atlas_table = make_table(
    ["GEO Accession", "Tissue Coverage", "Samples"],
    [
        ["GSE5629", "Seedlings and whole plants", "24"],
        ["GSE5630", "Leaves (rosette, cauline, senescing)", "60"],
        ["GSE5631", "Roots", "21"],
        ["GSE5632", "Flowers and pollen (all stages)", "66"],
        ["GSE5633", "Shoots, stems, and shoot apex", "42"],
        ["GSE5634", "Siliques and seeds (all stages)", "24"],
    ],
    [120, 260, 80]
)
story.append(atlas_table)
story.append(Spacer(1, 8))

story.append(Paragraph(
    "In total, this gives us 237 tissue-specific expression profiles across 20,833 genes. "
    "We organize these into a <b>hierarchical tissue structure</b>:", styles["Body"]))

hierarchy_table = make_table(
    ["Broad Organ", "Sub-Tissues", "Samples"],
    [
        ["Root", "root", "21"],
        ["Seedling", "seedling_green_parts, whole_plant_pre_bolting", "24"],
        ["Leaf_Shoot", "cotyledon, hypocotyl, rosette_leaf, cauline_leaf, senescing_leaf, rosette_vegetative, stem, shoot_apex (vegetative/transition)", "78"],
        ["Flower", "sepal, petal, stamen, carpel, mature_pollen, pedicel, flower_stage_9/10/12/15, inflorescence_apex", "90"],
        ["Seed_Silique", "silique_stage_3/4/5, seed_stage_6/7/8/9/10", "24"],
    ],
    [80, 300, 80]
)
story.append(hierarchy_table)
story.append(Spacer(1, 8))

story.append(Paragraph("3.2 The Tau Tissue-Specificity Index", styles["SubHead"]))
story.append(Paragraph(
    "To quantify how tissue-specific a gene's expression is, we compute the <b>Tau index</b>, "
    "a standard metric used in genomics. Tau ranges from 0 to 1:", styles["Body"]))

story.append(code_block(
    "Tau = sum(1 - x_i / max(x)) / (n - 1)\n\n"
    "where:\n"
    "  x_i = expression of the gene in tissue i\n"
    "  n   = number of tissues\n"
    "  max(x) = highest expression across all tissues"
))

story.append(Paragraph(
    "<b>Tau = 0</b> means the gene is expressed equally in all tissues (ubiquitous/constitutive). "
    "<b>Tau = 1</b> means the gene is expressed in only one tissue (perfectly tissue-specific). "
    "We use a threshold of <b>Tau &ge; 0.6</b> to classify a gene as tissue-specific; genes "
    "below this threshold are labeled 'constitutive' (broadly expressed).", styles["Body"]))

story.append(callout_box(
    "INTERMEDIATE NOTE: Why Tau and not just 'highest expression tissue'?",
    "Simply assigning a gene to the tissue where it is most expressed would be misleading for "
    "housekeeping genes that are expressed everywhere at similar levels. The Tau index first "
    "asks 'is this gene actually tissue-specific?' before making an assignment. A gene with "
    "Tau = 0.15 (expressed nearly everywhere) should not be assigned to any single tissue, "
    "even if one tissue happens to have marginally higher expression. Tau separates truly "
    "tissue-specific genes from broadly expressed ones."
))

story.append(callout_box(
    "ADVANCED NOTE: Limitations of the atlas approach",
    "This method infers tissue relevance from where genes are normally expressed in unstressed "
    "plants. It does not measure where the radiation response actually occurs. A gene that is "
    "normally root-specific might be induced in leaves by radiation. The atlas tells us the "
    "'default' tissue context of each gene, not the treatment-specific context. For direct "
    "tissue-level radiation responses, you would need to irradiate and harvest tissues "
    "separately (as in some newer experiments). Also, the ATH1 array covers ~21,000 of "
    "~27,000 Arabidopsis genes; genes not on the array are flagged 'unannotated'."
))

story.append(PageBreak())

print("Sections 1-3 added. Continuing...")


Sections 1-3 added. Continuing...


Add section 4 code walkthrough to PDF

In [47]:

# ── SECTION 4: Code Walkthrough ────────────────────────────────────────────────
story.append(Paragraph("4. Code Walkthrough: Section-by-Section Explanation", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "This section walks through the R script (<font face='Courier'>tissue_specific_deg_analysis.R</font>) "
    "block by block. The script is organized into 13 numbered sections. We explain what each "
    "section does, why it does it that way, and what the key R functions mean.", styles["Body"]))

# Section 0
story.append(Paragraph("4.1 Section 0: Package Installation and Loading", styles["SubHead"]))
story.append(Paragraph(
    "The script begins by checking whether all required R packages are installed and loading them. "
    "The key packages are:", styles["Body"]))

pkg_table = make_table(
    ["Package", "Purpose"],
    [
        ["GEOquery", "Downloads GEO datasets from NCBI"],
        ["ath1121501.db", "Maps ATH1 microarray probe IDs to Arabidopsis gene IDs"],
        ["AnnotationDbi", "Interface for querying annotation databases"],
        ["ComplexHeatmap", "Creates publication-quality heatmaps"],
        ["circlize", "Color mapping for heatmaps"],
        ["ggplot2", "General-purpose plotting (bar charts, volcano plots)"],
        ["UpSetR", "Creates UpSet diagrams for set intersection visualization"],
        ["gridExtra", "Arranges multiple plots on one page"],
    ],
    [120, 340]
)
story.append(pkg_table)
story.append(Spacer(1, 8))

story.append(code_block(
    '# Install missing packages automatically\n'
    'for (pkg in required_cran) {\n'
    '  if (!requireNamespace(pkg, quietly = TRUE))\n'
    '    install.packages(pkg, repos = "https://cloud.r-project.org")\n'
    '}'
))

story.append(Paragraph(
    "The <font face='Courier'>requireNamespace()</font> function checks if a package is available "
    "without loading it. If it returns FALSE, we install the package. This makes the script "
    "reproducible on any machine.", styles["Body"]))

# Section 1
story.append(Paragraph("4.2 Section 1: Loading and Validating the DEG Dataset", styles["SubHead"]))
story.append(Paragraph(
    "We read the CSV file and rename the unnamed first and last columns. The first column "
    "contains gene IDs; the last contains the yes/no DEG flag.", styles["Body"]))

story.append(code_block(
    'deg <- read.csv(deg_file, header = TRUE, stringsAsFactors = FALSE)\n'
    'colnames(deg)[1] <- "gene_id"\n'
    'colnames(deg)[ncol(deg)] <- "deg_flag"'
))

story.append(Paragraph(
    "We also validate the AGI gene ID format. Nuclear genes follow the pattern AT{1-5}Gnnnnn. "
    "Genes starting with ATCG (chloroplast) or ATMG (mitochondria) are organellar genes that "
    "won't be in the AtGenExpress nuclear atlas — they are flagged as 'unannotated' later.", styles["Body"]))

story.append(code_block(
    '# Check for nuclear AGI IDs (AT1G through AT5G)\n'
    'valid_nuclear <- grepl("^AT[1-5]G[0-9]{5}$", deg$gene_id)\n'
    '# Add regulation direction\n'
    'deg$regulation <- ifelse(deg$deg_flag == "yes",\n'
    '  ifelse(deg$log2FoldChange > 0, "up", "down"), "non_DEG")'
))

story.append(callout_box(
    "BEGINNER'S CORNER: What does grepl() do?",
    "<font face='Courier'>grepl()</font> checks whether each string matches a pattern. The pattern "
    "<font face='Courier'>^AT[1-5]G[0-9]{5}$</font> means: start (^) with 'AT', then a digit 1-5, "
    "then 'G', then exactly 5 digits, then end ($). This matches valid nuclear gene IDs like "
    "AT3G27630 but rejects ATCG00090 (chloroplast) or ATMG01090 (mitochondria)."
))

# Section 2-3
story.append(Paragraph("4.3 Sections 2-3: Downloading AtGenExpress and Building Tissue Labels", styles["SubHead"]))
story.append(Paragraph(
    "We download six GEO series using <font face='Courier'>getGEO()</font> from the GEOquery package. "
    "Each series returns an ExpressionSet object containing the expression matrix and sample metadata. "
    "We then extract tissue labels from the sample metadata and map them to a hierarchical structure "
    "of broad organs and sub-tissues.", styles["Body"]))

story.append(code_block(
    '# Download a GEO series\n'
    'gse <- getGEO("GSE5629", destdir = geodir, getGPL = TRUE)\n\n'
    '# Extract tissue from the characteristics column\n'
    'extracted <- sub(".*Tissue:\\\\s*", "", vals)  # remove "Tissue: " prefix\n'
    'extracted <- sub("\\\\s*;.*", "", extracted)   # remove anything after ;'
))

story.append(Paragraph(
    "The <font face='Courier'>build_tissue_map()</font> function uses a series of if/else statements "
    "with <font face='Courier'>grepl()</font> to classify each raw tissue label (e.g., 'flowers stage "
    "12, sepals') into a broad organ ('Flower') and sub-tissue ('sepal'). This is a rule-based "
    "mapping because the AtGenExpress tissue labels are free-text descriptions, not controlled "
    "vocabulary.", styles["Body"]))

# Section 4
story.append(Paragraph("4.4 Section 4: Mapping Microarray Probes to Gene IDs", styles["SubHead"]))
story.append(Paragraph(
    "The ATH1 microarray uses probe set IDs (e.g., 244901_at), not gene names. We use the "
    "<font face='Courier'>ath1121501.db</font> annotation package to map probes to AGI gene IDs "
    "via the TAIR column. Some genes have multiple probes; we keep the one with the highest "
    "average expression.", styles["Body"]))

story.append(code_block(
    '# Map probes to AGI gene IDs\n'
    'agi_map <- AnnotationDbi::select(ath1121501.db,\n'
    '  keys = probe_ids, columns = c("PROBEID", "TAIR"), keytype = "PROBEID")\n\n'
    '# Keep only nuclear AGI IDs\n'
    'agi_map <- agi_map[grepl("^AT[1-5]G[0-9]{5}", agi_map$TAIR), ]'
))

# Section 5
story.append(Paragraph("4.5 Section 5: Building Tissue-Level Expression Matrices", styles["SubHead"]))
story.append(Paragraph(
    "We average expression across all biological replicates within each tissue group, producing "
    "one expression value per gene per tissue. This is done at both the sub-tissue level (33 tissues) "
    "and the broad-organ level (5 organs).", styles["Body"]))

story.append(code_block(
    '# Average expression per sub-tissue\n'
    'sub_tissue_expr <- sapply(sub_tissues, function(st) {\n'
    '  samples_st <- all_samples$sample_id[all_samples$sub_tissue == st]\n'
    '  rowMeans(expr_mapped[, samples_st, drop = FALSE], na.rm = TRUE)\n'
    '})'
))

# Section 6
story.append(Paragraph("4.6 Section 6: Computing the Tau Index", styles["SubHead"]))
story.append(Paragraph(
    "The <font face='Courier'>compute_tau()</font> function applies the Tau formula to each gene "
    "(each row of the expression matrix). The <font face='Courier'>apply()</font> function with "
    "MARGIN = 1 iterates over rows.", styles["Body"]))

story.append(code_block(
    'compute_tau <- function(expr_matrix) {\n'
    '  expr_matrix[expr_matrix < 0] <- 0  # clamp negatives\n'
    '  apply(expr_matrix, 1, function(x) {\n'
    '    mx <- max(x, na.rm = TRUE)\n'
    '    if (mx == 0) return(NA)\n'
    '    n <- sum(!is.na(x))\n'
    '    sum(1 - (x / mx), na.rm = TRUE) / (n - 1)\n'
    '  })\n'
    '}'
))

story.append(Paragraph(
    "After computing Tau, we classify each gene as 'tissue_specific' (Tau &ge; 0.6), "
    "'constitutive' (Tau &lt; 0.6), or 'unannotated' (not in the atlas). We also identify "
    "the predominant tissue for each gene (the tissue with the highest expression).", styles["Body"]))

# Section 7
story.append(Paragraph("4.7 Section 7: Merging DEG Data with Tissue Annotations", styles["SubHead"]))
story.append(Paragraph(
    "We merge the DESeq2 results with the tissue annotations using <font face='Courier'>merge()</font>, "
    "which joins two data frames by a shared column (gene_id). Genes not in the atlas are flagged "
    "as 'unannotated'.", styles["Body"]))

# Sections 8-9
story.append(Paragraph("4.8 Sections 8-9: Exporting CSVs and Summary Table", styles["SubHead"]))
story.append(Paragraph(
    "We create a hierarchical folder structure and write one CSV per tissue. The summary table "
    "aggregates DEG counts, up/down breakdown, and statistics per tissue.", styles["Body"]))

# Sections 10-13
story.append(Paragraph("4.9 Sections 10-13: Generating Visualizations", styles["SubHead"]))
story.append(Paragraph(
    "Four figures are generated: (1) a stacked bar chart of DEG counts per organ, (2) volcano "
    "plots per organ, (3) a heatmap of the top 50 DEGs across all sub-tissues, and (4) an UpSet "
    "diagram showing organ expression overlaps. Each uses ggplot2 or ComplexHeatmap with "
    "colorblind-friendly palettes.", styles["Body"]))

story.append(PageBreak())

print("Section 4 added. Continuing...")


Section 4 added. Continuing...


Add sections 5-8 and build the PDF

In [49]:

# ── SECTION 5: Understanding the Outputs ───────────────────────────────────────
story.append(Paragraph("5. Understanding the Outputs: CSVs, Summary Table, and Figures", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("5.1 Hierarchical CSV Files", styles["SubHead"]))
story.append(Paragraph(
    "The script creates a folder called <font face='Courier'>tissue_specific_degs/</font> with "
    "the following structure:", styles["Body"]))

story.append(code_block(
    "tissue_specific_degs/\n"
    "  ├── Flower/\n"
    "  │   ├── Flower_all_DEGs.csv        (all flower-specific DEGs)\n"
    "  │   ├── carpel_DEGs.csv            (DEGs specific to carpels)\n"
    "  │   ├── petal_DEGs.csv\n"
    "  │   ├── sepal_DEGs.csv\n"
    "  │   ├── stamen_DEGs.csv\n"
    "  │   ├── mature_pollen_DEGs.csv\n"
    "  │   └── ... (one per flower sub-tissue)\n"
    "  ├── Leaf_Shoot/\n"
    "  │   ├── Leaf_Shoot_all_DEGs.csv\n"
    "  │   ├── rosette_leaf_DEGs.csv\n"
    "  │   ├── senescing_leaf_DEGs.csv\n"
    "  │   └── ...\n"
    "  ├── Root/\n"
    "  │   └── root_DEGs.csv\n"
    "  ├── Seed_Silique/\n"
    "  │   └── ... (seed and silique sub-tissues)\n"
    "  ├── Seedling/\n"
    "  │   └── ...\n"
    "  ├── constitutive_DEGs.csv          (broadly expressed DEGs)\n"
    "  ├── unannotated_DEGs.csv           (organellar / not on ATH1 array)\n"
    "  └── all_degs_with_tissue_annotation.csv  (master file: all 23,573 genes)"
))

story.append(Paragraph(
    "Each CSV contains the original DESeq2 columns plus the tissue annotations:", styles["Body"]))

output_table = make_table(
    ["Added Column", "Description"],
    [
        ["regulation", "up, down, or non_DEG"],
        ["tau_subtissue", "Tau index at sub-tissue level (0-1)"],
        ["tau_broad", "Tau index at broad-organ level (0-1)"],
        ["specificity", "tissue_specific, constitutive, or unannotated"],
        ["predominant_broad_organ", "Organ with highest expression (e.g., Flower)"],
        ["predominant_subtissue", "Sub-tissue with highest expression (e.g., sepal)"],
    ],
    [160, 300]
)
story.append(output_table)
story.append(Spacer(1, 8))

story.append(Paragraph("5.2 Summary Table (tissue_deg_summary.csv)", styles["SubHead"]))
story.append(Paragraph(
    "This table provides a bird's-eye view of the tissue distribution of radiation-responsive DEGs. "
    "Here are the key results from this dataset:", styles["Body"]))

summary_table = make_table(
    ["Broad Organ", "Total DEGs", "Up", "Down", "Median log2FC"],
    [
        ["Leaf_Shoot", "1,399", "727", "672", "+0.22"],
        ["Flower", "1,192", "677", "515", "+0.22"],
        ["Seed_Silique", "609", "437", "172", "+0.24"],
        ["Root", "432", "154", "278", "-0.25"],
        ["Seedling", "72", "54", "18", "+0.69"],
        ["Constitutive", "2,115", "1,055", "1,060", "-0.06"],
        ["Unannotated", "1,123", "670", "453", "+0.21"],
        ["TOTAL", "6,942", "3,719", "3,223", "+0.12"],
    ],
    [100, 80, 60, 60, 100]
)
story.append(summary_table)
story.append(Spacer(1, 8))

story.append(Paragraph(
    "A striking pattern emerges: <b>root-specific DEGs are predominantly downregulated</b> "
    "(278 down vs 154 up, median log2FC = -0.25), while <b>seed/silique and seedling DEGs are "
    "predominantly upregulated</b>. This suggests radiation suppresses root gene expression more "
    "than it enhances it, while developmental and reproductive tissues show more activation.", styles["Body"]))

story.append(Paragraph("5.3 Figures", styles["SubHead"]))

story.append(Paragraph("<b>Figure 1: Bar Chart (fig1_deg_counts_by_organ.png)</b> — "
    "A stacked bar chart showing the number of upregulated (blue) and downregulated (orange) "
    "DEGs in each broad organ. This gives a quick visual overview of which tissues have the "
    "most radiation-responsive genes and whether the response is primarily activation or "
    "suppression.", styles["Body"]))

story.append(Paragraph("<b>Figure 2: Volcano Plots (fig2_volcano_plots_by_organ.png)</b> — "
    "Five panels (one per organ), each showing log2FoldChange on the x-axis and -log10(padj) "
    "on the y-axis. Points above the dashed line are statistically significant. Blue points "
    "are upregulated DEGs, orange are downregulated. Grey points are constitutive genes shown "
    "for context. Volcano plots help you see both the magnitude and significance of expression "
    "changes simultaneously.", styles["Body"]))

story.append(Paragraph("<b>Figure 3: Heatmap (fig3_heatmap_top_degs.png)</b> — "
    "Shows the tissue expression patterns of the 50 most significant DEGs across all 33 "
    "sub-tissues. Each row is a gene (z-scored expression), each column is a sub-tissue. "
    "The top annotation bar colors sub-tissues by broad organ. The right annotation shows "
    "whether each DEG is up- or downregulated in response to radiation. This reveals which "
    "tissues the most significant radiation genes normally operate in.", styles["Body"]))

story.append(Paragraph("<b>Figure 4: UpSet Diagram (fig4_upset_organ_overlaps.png)</b> — "
    "Shows how many DEGs are expressed above their median level in each combination of organs. "
    "The bar chart on the left shows the total number of DEGs expressed in each organ. The "
    "main bar chart shows intersection sizes — how many DEGs are co-expressed across multiple "
    "organs. Connected dots indicate which organs are in each intersection. This helps identify "
    "DEGs that are broadly active vs. organ-restricted.", styles["Body"]))

story.append(PageBreak())

# ── SECTION 6: How to Run the Code ─────────────────────────────────────────────
story.append(Paragraph("6. How to Run the Code: Prerequisites and Execution", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("6.1 Prerequisites", styles["SubHead"]))
story.append(Paragraph(
    "You need R (version 4.0 or later) with internet access (for downloading GEO data). "
    "The script will automatically install any missing packages.", styles["Body"]))

story.append(Paragraph("6.2 Step-by-Step Instructions", styles["SubHead"]))

story.append(Paragraph("<b>Step 1:</b> Place the DEG CSV file and the R script in the same directory.", styles["Body"]))
story.append(code_block(
    "# Your directory should look like:\n"
    "# /your_project/\n"
    "#   ├── DEG_OSD498_510_radiation_effect.csv\n"
    "#   └── tissue_specific_deg_analysis.R"
))

story.append(Paragraph("<b>Step 2:</b> Open R or RStudio and set the working directory to "
    "the folder containing your files.", styles["Body"]))
story.append(code_block(
    'setwd("/path/to/your_project")'
))

story.append(Paragraph("<b>Step 3:</b> Run the script.", styles["Body"]))
story.append(code_block(
    '# In RStudio: click "Source" or press Cmd/Ctrl + Shift + S\n'
    '# In terminal:\n'
    'Rscript tissue_specific_deg_analysis.R'
))

story.append(Paragraph("<b>Step 4:</b> Wait for the analysis to complete. The GEO download "
    "step may take 5-15 minutes depending on your internet speed. The entire analysis "
    "typically completes in 10-20 minutes.", styles["Body"]))

story.append(Paragraph("<b>Step 5:</b> Check the outputs. All files will be created in your "
    "working directory:", styles["Body"]))

story.append(code_block(
    "# Output files:\n"
    "#   tissue_specific_degs/          (folder with hierarchical CSVs)\n"
    "#   tissue_deg_summary.csv         (summary table)\n"
    "#   fig1_deg_counts_by_organ.png\n"
    "#   fig2_volcano_plots_by_organ.png\n"
    "#   fig3_heatmap_top_degs.png\n"
    "#   fig4_upset_organ_overlaps.png"
))

story.append(callout_box(
    "BEGINNER'S CORNER: What is RStudio?",
    "RStudio is a free, user-friendly interface for R. Instead of typing commands in a plain "
    "terminal, you get a code editor (with syntax highlighting), a console (where R runs), "
    "an environment panel (showing your variables), and a plots panel (showing your figures). "
    "Download it from posit.co. To run a script, open it in RStudio and click the 'Source' "
    "button in the top right of the code editor."
))

story.append(Paragraph("6.3 Customizing the Analysis", styles["SubHead"]))
story.append(Paragraph(
    "You can modify these parameters near the top of the script:", styles["Body"]))

params_table = make_table(
    ["Parameter", "Default", "Effect of Changing"],
    [
        ["deg_file", "DEG_OSD498_510_...", "Point to a different DEG file"],
        ["TAU_THRESHOLD", "0.6", "Lower (e.g., 0.5) = more genes called tissue-specific; Higher (e.g., 0.8) = stricter"],
        ["UP_COLOR / DOWN_COLOR", "#0072B2 / #D55E00", "Change plot colors"],
    ],
    [130, 130, 200]
)
story.append(params_table)

story.append(PageBreak())

# ── SECTION 7: Interpretation Guide ────────────────────────────────────────────
story.append(Paragraph("7. Interpretation Guide: What Tissue-Specific Radiation Responses Mean", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("7.1 Key Findings from This Dataset", styles["SubHead"]))

story.append(Paragraph(
    "<b>Finding 1: Leaf/shoot tissues harbor the most radiation-responsive DEGs (1,399).</b> "
    "This is consistent with leaves being the primary site of photosynthesis and oxidative "
    "metabolism. Radiation generates reactive oxygen species (ROS), and leaf tissues have "
    "extensive antioxidant systems that may be transcriptionally modulated.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 2: Root-specific DEGs are predominantly downregulated (64%).</b> "
    "Radiation appears to suppress root gene expression more than it activates it. This could "
    "reflect cell cycle arrest (roots are actively growing and dividing, making them vulnerable "
    "to DNA damage checkpoints) or a reallocation of resources away from root growth.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 3: Senescing leaf DEGs are almost entirely upregulated (97%, 426/436).</b> "
    "Radiation may accelerate senescence programs in leaves. Senescence involves the ordered "
    "breakdown and remobilization of cellular components, and radiation-induced DNA damage "
    "could trigger premature activation of these pathways.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 4: Seed/silique DEGs are predominantly upregulated (72%).</b> "
    "Reproductive tissues may activate protective pathways to shield developing embryos from "
    "radiation damage. This is consistent with the importance of protecting the germline in "
    "all organisms.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 5: 2,115 DEGs are constitutive (broadly expressed).</b> "
    "These are housekeeping-like genes that respond to radiation regardless of tissue context. "
    "They likely include core DNA damage response genes (e.g., PARP, BRCA homologs, DNA "
    "polymerases) that are needed in every cell type.", styles["Body"]))

story.append(Paragraph("7.2 Biological Interpretation Framework", styles["SubHead"]))
story.append(Paragraph(
    "When interpreting tissue-specific DEG results, consider these questions:", styles["Body"]))

story.append(Paragraph("&bull; <b>Is the direction consistent with known biology?</b> "
    "If root growth is known to be inhibited by radiation, downregulation of root-specific "
    "growth genes makes sense.", styles["BulletStyle"]))
story.append(Paragraph("&bull; <b>Are DNA repair genes tissue-specific or constitutive?</b> "
    "Core repair genes should be constitutive (every cell needs to repair DNA). Tissue-specific "
    "repair genes might reflect different damage types or repair strategies in different organs.", styles["BulletStyle"]))
story.append(Paragraph("&bull; <b>Do stress response genes show tissue specificity?</b> "
    "Leaf-specific stress genes might relate to oxidative stress (from photosynthesis + radiation), "
    "while root-specific stress genes might relate to different stress pathways.", styles["BulletStyle"]))
story.append(Paragraph("&bull; <b>Are developmental genes affected?</b> "
    "If flower or seed developmental genes are disrupted, this could have implications for "
    "plant reproduction in space environments.", styles["BulletStyle"]))

story.append(callout_box(
    "ADVANCED NOTE: Caveats and limitations",
    "1. The AtGenExpress atlas measures baseline expression in unstressed plants. Radiation "
    "may alter tissue-specificity. 2. The ATH1 array covers ~21,000 of ~27,000 genes; ~1,123 "
    "DEGs (mostly organellar) could not be annotated. 3. The Tau threshold of 0.6 is a "
    "convention, not a biological absolute — results may shift with different thresholds. "
    "4. The original experiment used whole seedlings; true tissue-level radiation responses "
    "require tissue-specific irradiation and harvest. 5. Cross-platform differences between "
    "microarray (atlas) and RNA-seq (DEG data) may introduce noise."
))

story.append(PageBreak())

# ── SECTION 8: Exercises and Extensions ────────────────────────────────────────
story.append(Paragraph("8. Exercises and Extensions for Further Exploration", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "This section provides exercises for students to deepen their understanding and suggestions "
    "for extending the analysis.", styles["Body"]))

story.append(Paragraph("8.1 Beginner Exercises", styles["SubHead"]))

story.append(Paragraph("&bull; <b>Exercise 1:</b> Open <font face='Courier'>Root/root_DEGs.csv</font> "
    "in a spreadsheet. Sort by log2FoldChange. What are the top 5 upregulated and top 5 "
    "downregulated root-specific genes? Look up their functions on TAIR "
    "(www.arabidopsis.org).", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 2:</b> Compare the number of DEGs in "
    "<font face='Courier'>Flower/mature_pollen_DEGs.csv</font> vs "
    "<font face='Courier'>Flower/sepal_DEGs.csv</font>. Why might pollen have more "
    "radiation-responsive genes than sepals?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 3:</b> Look at the bar chart (Figure 1). Which organ "
    "has the most balanced up/down ratio? Which is most skewed? What might this tell you "
    "about how that tissue responds to radiation?", styles["BulletStyle"]))

story.append(Paragraph("8.2 Intermediate Exercises", styles["SubHead"]))

story.append(Paragraph("&bull; <b>Exercise 4:</b> Modify the TAU_THRESHOLD to 0.5 and re-run "
    "the script. How does this change the number of tissue-specific vs constitutive DEGs? "
    "Is the change dramatic or modest? What does this tell you about the sensitivity of "
    "the results to the threshold?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 5:</b> Extract the constitutive DEGs and perform "
    "Gene Ontology enrichment analysis (using R packages like clusterProfiler or online tools "
    "like DAVID). Are DNA repair genes overrepresented in the constitutive set?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 6:</b> Create a Venn diagram comparing the DEG sets "
    "from two organs (e.g., Root vs Flower). How many DEGs are tissue-specific to both? "
    "(Hint: since each gene is assigned to one tissue, you'll need to use the expression "
    "threshold approach from the UpSet diagram.)", styles["BulletStyle"]))

story.append(Paragraph("8.3 Advanced Extensions", styles["SubHead"]))

story.append(Paragraph("&bull; <b>Extension 1:</b> Integrate Gene Ontology (GO) enrichment "
    "analysis for each tissue-specific DEG set. Use the <font face='Courier'>clusterProfiler</font> "
    "R package with the Arabidopsis org.At.tair.db annotation. This will tell you which "
    "biological processes are enriched in each tissue's radiation response.", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Extension 2:</b> Compare these results with other NASA GeneLab "
    "radiation studies (OSD-502, OSD-508, OSD-658). Do the same tissue-specific patterns hold "
    "across different radiation types (gamma vs HZE vs simulated GCR)?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Extension 3:</b> Use the newer Arabidopsis single-cell atlas "
    "(GSE226097, published in Nature Plants 2025) instead of AtGenExpress for cell-type-level "
    "resolution. This would give you much finer granularity (individual cell types rather than "
    "bulk tissues).", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Extension 4:</b> Build a gene regulatory network for each "
    "tissue-specific DEG set using the SCENIC or WGCNA approach. This would identify "
    "transcription factors driving the tissue-specific radiation responses.", styles["BulletStyle"]))

story.append(divider())
story.append(Spacer(1, 20))

story.append(Paragraph(
    "<b>References:</b>", styles["SubHead"]))
story.append(Paragraph(
    "1. Schmid M et al. (2005) A gene expression map of Arabidopsis thaliana development. "
    "Nature Genetics 37:501-506.", styles["Body"]))
story.append(Paragraph(
    "2. Yanai I et al. (2005) Genome-wide midrange transcription profiles reveal expression "
    "level relationships in human tissue specification. Bioinformatics 21:650-659. (Tau index)", styles["Body"]))
story.append(Paragraph(
    "3. Love MI, Huber W, Anders S (2014) Moderated estimation of fold change and dispersion "
    "for RNA-seq data with DESeq2. Genome Biology 15:550.", styles["Body"]))
story.append(Paragraph(
    "4. NASA GeneLab OSD-498 and OSD-510: https://osdr.nasa.gov/bio/repo/data/studies/OSD-498", styles["Body"]))

# ── Build the PDF ──
doc.build(story, onFirstPage=page_header_footer, onLaterPages=page_header_footer)

# Validate
from pypdf import PdfReader
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)
first_page_text = reader.pages[0].extract_text()

print(f"PDF generated: {output_path}")
print(f"Pages: {page_count}")
print(f"File size: {file_size:,} bytes")
print(f"First page text length: {len(first_page_text.strip())} chars")
assert page_count >= 2, "Too few pages"
assert file_size > 5000, "File too small"
assert len(first_page_text.strip()) > 0, "No text on first page"
print("Validation passed!")


PDF generated: /mnt/results/instruction_manual_tissue_specific_deg.pdf
Pages: 17
File size: 40,355 bytes
First page text length: 1271 chars
Validation passed!
